In [ ]:
# packages
import math
import folium
import pandas as pd
import geopandas as gpd
import contextily as ctx
from pyproj import Transformer
import matplotlib.pyplot as plt
from shapely.geometry import Point
from matplotlib.lines import Line2D
import matplotlib.patheffects as pe
from matplotlib.patches import Rectangle, ConnectionPatch
from matplotlib.ticker import FuncFormatter, MultipleLocator
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset


In [ ]:
# Path to your KML file
kml_path = "/bsuhome/tnde/geoscience/albedo_downscaling/shape_files/East_River.kml"
# kml_path = "/bsuhome/tnde/geoscience/albedo_downscaling/shape_files/Colorado_River_Basin_Hydrological_Boundaries.kml"

# Read the KML
study_area = gpd.read_file(kml_path)

# Check the data
print(study_area.head())
print(study_area.crs)

# Plot the study area
fig, ax = plt.subplots(figsize=(8, 8))

study_area.plot(
    ax=ax,
    facecolor="none",
    edgecolor="red",
    linewidth=2
)

ax.set_title("Study Area Boundary")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_aspect("equal")

plt.show()

In [ ]:
kml_path = "/bsuhome/tnde/geoscience/albedo_downscaling/shape_files/East_River.kml"
# kml_path = "/bsuhome/tnde/geoscience/albedo_downscaling/shape_files/Colorado_River_Basin_Hydrological_Boundaries.kml"

# List available layers
layers = gpd.list_layers(kml_path)
print(layers)

In [ ]:
study_area = gpd.read_file(kml_path, layer="East_River")

fig, ax = plt.subplots(figsize=(8, 8))
study_area.plot(ax=ax, facecolor="none", edgecolor="red", linewidth=2)

ax.set_title("Study Area Boundary")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_aspect("equal")

plt.show()

In [ ]:
# ============================================================
# KML file path and layer names
# ============================================================

kml_path = "/bsuhome/tnde/geoscience/albedo_downscaling/shape_files/East_River.kml"

kml_layers = {
    "East_River.kml": "red",
    "East_River": "blue"
}


# ============================================================
# Read both KML layers
# ============================================================

study_layers = {}

for layer_name, color in kml_layers.items():

    gdf = gpd.read_file(kml_path, layer=layer_name)

    # Remove missing or empty geometries
    gdf = gdf[
        gdf.geometry.notnull() & ~gdf.geometry.is_empty
    ].copy()

    # Your KML is already lon/lat.
    # This only labels the coordinates if GeoPandas did not detect the CRS.
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326")

    study_layers[layer_name] = {
        "gdf": gdf,
        "color": color
    }

    print(f"\nLayer: {layer_name}")
    print("CRS:", gdf.crs)
    print("Geometry types:", gdf.geometry.geom_type.unique())
    print("Bounds:", gdf.total_bounds)


# ============================================================
# Helper function for plotting different geometry types
# ============================================================

def plot_kml_layer(ax, gdf, color, linewidth=2.5, label=None):
    """
    Plot one KML layer.
    Polygons are plotted as boundaries.
    Lines are plotted as lines.
    Points are plotted as points.
    """

    polygon_types = ["Polygon", "MultiPolygon"]
    line_types = ["LineString", "MultiLineString", "LinearRing"]
    point_types = ["Point", "MultiPoint"]

    # Plot polygon boundaries
    poly_gdf = gdf[gdf.geometry.geom_type.isin(polygon_types)]
    if not poly_gdf.empty:
        poly_gdf.boundary.plot(
            ax=ax,
            color=color,
            linewidth=linewidth,
            label=label
        )

    # Plot line geometries
    line_gdf = gdf[gdf.geometry.geom_type.isin(line_types)]
    if not line_gdf.empty:
        line_gdf.plot(
            ax=ax,
            color=color,
            linewidth=linewidth,
            label=label
        )

    # Plot point geometries
    point_gdf = gdf[gdf.geometry.geom_type.isin(point_types)]
    if not point_gdf.empty:
        point_gdf.plot(
            ax=ax,
            color=color,
            markersize=35,
            label=label
        )

    # Plot any remaining geometry types, such as GeometryCollection
    other_gdf = gdf[
        ~gdf.geometry.geom_type.isin(polygon_types + line_types + point_types)
    ]
    if not other_gdf.empty:
        other_gdf.plot(
            ax=ax,
            facecolor="none",
            edgecolor=color,
            color=color,
            linewidth=linewidth,
            label=label
        )


# ============================================================
# Plot both KML layers only
# ============================================================

fig, ax = plt.subplots(figsize=(8, 8))

for layer_name, info in study_layers.items():
    plot_kml_layer(
        ax=ax,
        gdf=info["gdf"],
        color=info["color"],
        linewidth=2.5,
        label=layer_name
    )

# Axis labels
ax.set_xlabel("Longitude", fontsize=14)
ax.set_ylabel("Latitude", fontsize=14)

# Title
ax.set_title("KML Layers: East River Study Area", fontsize=16)

# Make the map shape display correctly
ax.set_aspect("equal")

# Add grid
ax.grid(True, alpha=0.3)

# Tick font size
ax.tick_params(axis="both", labelsize=12)

# Legend
legend_handles = [
    Line2D([0], [0], color="red", lw=2.5, label="East River Basin"),
    Line2D([0], [0], color="blue", lw=2.5, label="East River")
]

ax.legend(
    handles=legend_handles,
    loc="best",
    fontsize=11,
    frameon=True
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# KML file path and layer names
# ============================================================

kml_path = "/bsuhome/tnde/geoscience/albedo_downscaling/shape_files/East_River.kml"

kml_layers = {
    "East_River.kml": "red",
    "East_River": "blue"
}


# ============================================================
# Read both KML layers
# ============================================================

study_layers = {}

for layer_name, color in kml_layers.items():

    gdf = gpd.read_file(kml_path, layer=layer_name)

    # Remove missing or empty geometries
    gdf = gdf[
        gdf.geometry.notnull() & ~gdf.geometry.is_empty
    ].copy()

    # KML is already lon/lat
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326")

    study_layers[layer_name] = {
        "gdf": gdf,
        "color": color
    }

    print(f"\nLayer: {layer_name}")
    print("CRS:", gdf.crs)
    print("Geometry types:", gdf.geometry.geom_type.unique())
    print("Bounds:", gdf.total_bounds)


# ============================================================
# SAIL locations in decimal degrees
# ============================================================

# 38°57'22.35"N, 106°59'16.66"W
sail_road_lon = -106.987961
sail_road_lat = 38.956208

# 38°57'22.99"N, 106°59'8.79"W
sail_hill_lon = -106.985775
sail_hill_lat = 38.956386

locations_gdf = gpd.GeoDataFrame(
    {
        "name": [
            "SAIL site near County Road 317",
            "Field instruments on adjacent hill"
        ]
    },
    geometry=[
        Point(sail_road_lon, sail_road_lat),
        Point(sail_hill_lon, sail_hill_lat)
    ],
    crs="EPSG:4326"
)


# ============================================================
# Helper function for plotting different geometry types
# ============================================================

def plot_kml_layer(ax, gdf, color, linewidth=2.5, label=None):
    """
    Plot one KML layer.
    Polygons are plotted as boundaries.
    Lines are plotted as lines.
    Points are plotted as points.
    """

    polygon_types = ["Polygon", "MultiPolygon"]
    line_types = ["LineString", "MultiLineString", "LinearRing"]
    point_types = ["Point", "MultiPoint"]

    poly_gdf = gdf[gdf.geometry.geom_type.isin(polygon_types)]
    if not poly_gdf.empty:
        poly_gdf.boundary.plot(
            ax=ax,
            color=color,
            linewidth=linewidth,
            label=label
        )

    line_gdf = gdf[gdf.geometry.geom_type.isin(line_types)]
    if not line_gdf.empty:
        line_gdf.plot(
            ax=ax,
            color=color,
            linewidth=linewidth,
            label=label
        )

    point_gdf = gdf[gdf.geometry.geom_type.isin(point_types)]
    if not point_gdf.empty:
        point_gdf.plot(
            ax=ax,
            color=color,
            markersize=35,
            label=label
        )

    other_gdf = gdf[
        ~gdf.geometry.geom_type.isin(polygon_types + line_types + point_types)
    ]
    if not other_gdf.empty:
        other_gdf.plot(
            ax=ax,
            facecolor="none",
            edgecolor=color,
            color=color,
            linewidth=linewidth,
            label=label
        )


# ============================================================
# Plot both KML layers and overlay the two SAIL locations
# ============================================================

fig, ax = plt.subplots(figsize=(8, 8))

# Plot KML layers
for layer_name, info in study_layers.items():
    plot_kml_layer(
        ax=ax,
        gdf=info["gdf"],
        color=info["color"],
        linewidth=2.5,
        label=layer_name
    )

# Plot SAIL locations
ax.scatter(
    [sail_road_lon, sail_hill_lon],
    [sail_road_lat, sail_hill_lat],
    color=["black", "magenta"],
    s=80,
    marker="o",
    zorder=10
)

# Add labels for the two locations
ax.annotate(
    "Road 317 site",
    xy=(sail_road_lon, sail_road_lat),
    xytext=(8, 8),
    textcoords="offset points",
    fontsize=11,
    color="black"
)

ax.annotate(
    "Adjacent hill site",
    xy=(sail_hill_lon, sail_hill_lat),
    xytext=(8, -14),
    textcoords="offset points",
    fontsize=11,
    color="magenta"
)

# Axis labels
ax.set_xlabel("Longitude", fontsize=14)
ax.set_ylabel("Latitude", fontsize=14)

# Title
ax.set_title("KML Layers with SAIL Locations", fontsize=16)

# Keep map aspect
ax.set_aspect("equal")

# Grid
ax.grid(True, alpha=0.3)

# Tick font size
ax.tick_params(axis="both", labelsize=12)

# Legend
legend_handles = [
    Line2D([0], [0], color="red", lw=2.5, label="East_River Basin"),
    Line2D([0], [0], color="blue", lw=2.5, label="East River"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="black",
           markersize=9, label="SAIL location"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="magenta",
           markersize=9, label="Field Pyranometer")
]

ax.legend(
    handles=legend_handles,
    loc="best",
    fontsize=11,
    frameon=True
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# KML file path and layer names
# ============================================================

kml_path = "/bsuhome/tnde/geoscience/albedo_downscaling/shape_files/East_River.kml"

kml_layers = {
    "East_River.kml": "red",
    "East_River": "blue"
}

# ============================================================
# Read both KML layers in lon/lat
# ============================================================

study_layers = {}

for layer_name, color in kml_layers.items():

    gdf = gpd.read_file(kml_path, layer=layer_name)

    gdf = gdf[
        gdf.geometry.notnull() & ~gdf.geometry.is_empty
    ].copy()

    # KML is already lon/lat.
    # Only assign CRS if GeoPandas did not detect it.
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326")

    study_layers[layer_name] = {
        "gdf": gdf,
        "color": color
    }

    print(f"\nLayer: {layer_name}")
    print("CRS:", gdf.crs)
    print("Geometry types:", gdf.geometry.geom_type.unique())
    print("Bounds:", gdf.total_bounds)


# ============================================================
# SAIL locations in lon/lat
# ============================================================

# 38°57'22.35"N, 106°59'16.66"W
sail_road_lon = -106.987961
sail_road_lat = 38.956208

# 38°57'22.99"N, 106°59'8.79"W
sail_hill_lon = -106.985775
sail_hill_lat = 38.956386

locations_gdf = gpd.GeoDataFrame(
    {
        "name": [
            "SAIL site near County Road 317",
            "Field instruments on adjacent hill"
        ]
    },
    geometry=[
        Point(sail_road_lon, sail_road_lat),
        Point(sail_hill_lon, sail_hill_lat)
    ],
    crs="EPSG:4326"
)


# ============================================================
# Reproject KML layers and points to Web Mercator for basemap
# ============================================================

basemap_crs = "EPSG:3857"

study_layers_3857 = {}

for layer_name, info in study_layers.items():
    study_layers_3857[layer_name] = {
        "gdf": info["gdf"].to_crs(basemap_crs),
        "color": info["color"]
    }

locations_3857 = locations_gdf.to_crs(basemap_crs)


# ============================================================
# Helper function for plotting KML geometry types
# ============================================================

def plot_kml_layer(ax, gdf, color, linewidth=6, linestyle="-", label=None):
    """
    Plot one KML layer.
    Polygons are plotted as boundaries.
    Lines are plotted as lines.
    Points are plotted as points.
    """

    polygon_types = ["Polygon", "MultiPolygon"]
    line_types = ["LineString", "MultiLineString", "LinearRing"]
    point_types = ["Point", "MultiPoint"]

    # Polygon boundaries
    poly_gdf = gdf[gdf.geometry.geom_type.isin(polygon_types)]
    if not poly_gdf.empty:
        poly_gdf.boundary.plot(
            ax=ax,
            color=color,
            linewidth=linewidth,
            linestyle=linestyle,
            zorder=5,
            label=label
        )

    # Lines
    line_gdf = gdf[gdf.geometry.geom_type.isin(line_types)]
    if not line_gdf.empty:
        line_gdf.plot(
            ax=ax,
            color=color,
            linewidth=linewidth,
            linestyle=linestyle,
            zorder=6,
            label=label
        )

    # Points
    point_gdf = gdf[gdf.geometry.geom_type.isin(point_types)]
    if not point_gdf.empty:
        point_gdf.plot(
            ax=ax,
            color=color,
            markersize=35,
            zorder=7,
            label=label
        )

    # Any other geometry type
    other_gdf = gdf[
        ~gdf.geometry.geom_type.isin(polygon_types + line_types + point_types)
    ]
    if not other_gdf.empty:
        other_gdf.plot(
            ax=ax,
            facecolor="none",
            edgecolor=color,
            color=color,
            linewidth=linewidth,
            linestyle=linestyle,
            zorder=8,
            label=label
        )


# ============================================================
# Plot KML layers with Esri World Imagery background
# ============================================================

fig, ax = plt.subplots(figsize=(10, 10))

# Plot KML layers
for layer_name, info in study_layers_3857.items():

    # Make the smaller boundary dashed
    linestyle = "--" if layer_name == "East_River" else "-"

    plot_kml_layer(
        ax=ax,
        gdf=info["gdf"],
        color=info["color"],
        linewidth=3.0,
        linestyle=linestyle,
        label=layer_name
    )

# Plot SAIL locations
locations_3857.plot(
    ax=ax,
    color=["blue", "magenta"],
    markersize=120,
    zorder=10
)

# Add location labels
for idx, row in locations_3857.iterrows():
    x = row.geometry.x
    y = row.geometry.y

    if "Road" in row["name"]:
        label = "SAIL site\nAdjacent Gunnison County Rd 317"
        color = "blue"
        xytext = (8, 8)
    else:
        label = "Field pyranometer\nAdjacent hill"
        color = "magenta"
        xytext = (8, -22)

    text = ax.annotate(
        label,
        xy=(x, y),
        xytext=xytext,
        textcoords="offset points",
        fontsize=14,
        color=color,
        zorder=11
    )

    # White outline so labels remain visible on imagery
    text.set_path_effects([
        pe.withStroke(linewidth=3, foreground="white")
    ])


# ============================================================
# Set map extent with padding
# ============================================================

# Combine bounds from all KML layers and locations
all_bounds = []

for info in study_layers_3857.values():
    all_bounds.append(info["gdf"].total_bounds)

all_bounds.append(locations_3857.total_bounds)

xmin = min(b[0] for b in all_bounds)
ymin = min(b[1] for b in all_bounds)
xmax = max(b[2] for b in all_bounds)
ymax = max(b[3] for b in all_bounds)

xpad = (xmax - xmin) * 0.08
ypad = (ymax - ymin) * 0.08

ax.set_xlim(xmin - xpad, xmax + xpad)
ax.set_ylim(ymin - ypad, ymax + ypad)


# ============================================================
# Add Esri World Imagery basemap
# ============================================================

ctx.add_basemap(
    ax,
    source=ctx.providers.Esri.WorldImagery,
    zoom=13,
    attribution_size=6
)


# ============================================================
# Final styling
# ============================================================

ax.set_title("East River Study Area with SAIL Locations", fontsize=20)

# Since the basemap is in Web Mercator, hide projected x/y axes
ax.set_axis_off()

legend_handles = [
    Line2D([0], [0], color="red", lw=3.0, label="East River Basin"),
    Line2D([0], [0], color="blue", lw=3.0, linestyle="--", label="East River"),
    Line2D(
        [0], [0],
        marker="o",
        color="w",
        markerfacecolor="blue",
        markersize=9,
        label="SAIL site"
    ),
    Line2D(
        [0], [0],
        marker="o",
        color="w",
        markerfacecolor="magenta",
        markersize=9,
        label="Field Pyranometer"
    )
]

ax.legend(
    handles=legend_handles,
    loc="lower right",
    fontsize=13,
    frameon=True,
    facecolor="white",
    framealpha=0.9
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# KML file path and layer names
# ============================================================

kml_path = "/bsuhome/tnde/geoscience/albedo_downscaling/shape_files/East_River.kml"

kml_layers = {
    "East_River.kml": "red",
    "East_River": "blue"
}


# ============================================================
# Read both KML layers in lon/lat
# ============================================================

study_layers = {}

for layer_name, color in kml_layers.items():

    gdf = gpd.read_file(kml_path, layer=layer_name)

    gdf = gdf[
        gdf.geometry.notnull() & ~gdf.geometry.is_empty
    ].copy()

    # KML is already lon/lat.
    # Only assign CRS if GeoPandas did not detect it.
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326")

    study_layers[layer_name] = {
        "gdf": gdf,
        "color": color
    }

    print(f"\nLayer: {layer_name}")
    print("CRS:", gdf.crs)
    print("Geometry types:", gdf.geometry.geom_type.unique())
    print("Bounds:", gdf.total_bounds)


# ============================================================
# SAIL locations in lon/lat
# ============================================================

# 38°57'22.35"N, 106°59'16.66"W
sail_road_lon = -106.987961
sail_road_lat = 38.956208

# 38°57'22.99"N, 106°59'8.79"W
sail_hill_lon = -106.985775
sail_hill_lat = 38.956386

locations_gdf = gpd.GeoDataFrame(
    {
        "name": [
            "SAIL site near County Road 317",
            "Field pyranometer on adjacent hill"
        ],
        "color": [
            "black",
            "magenta"
        ]
    },
    geometry=[
        Point(sail_road_lon, sail_road_lat),
        Point(sail_hill_lon, sail_hill_lat)
    ],
    crs="EPSG:4326"
)


# ============================================================
# Read Colorado and surrounding state boundaries
# ============================================================

states_url = "https://www2.census.gov/geo/tiger/GENZ2023/shp/cb_2023_us_state_500k.zip"

states = gpd.read_file(states_url)

state_names = [
    "Colorado",
    "Utah",
    "Wyoming",
    "Nebraska",
    "Kansas",
    "Oklahoma",
    "New Mexico",
    "Arizona"
]

states_region = states[states["NAME"].isin(state_names)].copy()


# ============================================================
# Reproject everything to Web Mercator for Esri basemap
# ============================================================

basemap_crs = "EPSG:3857"

study_layers_3857 = {}

for layer_name, info in study_layers.items():
    study_layers_3857[layer_name] = {
        "gdf": info["gdf"].to_crs(basemap_crs),
        "color": info["color"]
    }

locations_3857 = locations_gdf.to_crs(basemap_crs)
states_region_3857 = states_region.to_crs(basemap_crs)


# ============================================================
# Helper function for plotting KML geometry types
# ============================================================

def plot_kml_layer(ax, gdf, color, linewidth=6, linestyle="-", label=None):
    """
    Plot one KML layer.
    Polygons are plotted as boundaries.
    Lines are plotted as lines.
    Points are plotted as points.
    """

    polygon_types = ["Polygon", "MultiPolygon"]
    line_types = ["LineString", "MultiLineString", "LinearRing"]
    point_types = ["Point", "MultiPoint"]

    # Polygon boundaries
    poly_gdf = gdf[gdf.geometry.geom_type.isin(polygon_types)]
    if not poly_gdf.empty:
        poly_gdf.boundary.plot(
            ax=ax,
            color=color,
            linewidth=linewidth,
            linestyle=linestyle,
            zorder=30,
            label=label
        )

    # Lines
    line_gdf = gdf[gdf.geometry.geom_type.isin(line_types)]
    if not line_gdf.empty:
        line_gdf.plot(
            ax=ax,
            color=color,
            linewidth=linewidth,
            linestyle=linestyle,
            zorder=31,
            label=label
        )

    # Points
    point_gdf = gdf[gdf.geometry.geom_type.isin(point_types)]
    if not point_gdf.empty:
        point_gdf.plot(
            ax=ax,
            color=color,
            markersize=35,
            zorder=32,
            label=label
        )

    # Any other geometry type
    other_gdf = gdf[
        ~gdf.geometry.geom_type.isin(polygon_types + line_types + point_types)
    ]
    if not other_gdf.empty:
        other_gdf.plot(
            ax=ax,
            facecolor="none",
            edgecolor=color,
            color=color,
            linewidth=linewidth,
            linestyle=linestyle,
            zorder=33,
            label=label
        )


# ============================================================
# Create plot
# ============================================================

fig, ax = plt.subplots(figsize=(12, 10))


# ============================================================
# Set map extent to Colorado and surrounding states
# ============================================================

xmin, ymin, xmax, ymax = states_region_3857.total_bounds

xpad = (xmax - xmin) * 0.03
ypad = (ymax - ymin) * 0.03

ax.set_xlim(xmin - xpad, xmax + xpad)
ax.set_ylim(ymin - ypad, ymax + ypad)


# ============================================================
# Add Esri World Imagery basemap
# ============================================================

ctx.add_basemap(
    ax,
    source=ctx.providers.Esri.WorldImagery,
    zoom=6,
    attribution_size=6
)


# ============================================================
# Plot state boundaries
# ============================================================

# White outline underneath for contrast
states_region_3857.boundary.plot(
    ax=ax,
    color="white",
    linewidth=2.0,
    zorder=20
)

# Black state boundary line on top
states_region_3857.boundary.plot(
    ax=ax,
    color="black",
    linewidth=0.8,
    zorder=21
)


# ============================================================
# Add state labels
# ============================================================

for _, row in states_region_3857.iterrows():
    point = row.geometry.representative_point()

    text = ax.text(
        point.x,
        point.y,
        row["NAME"],
        fontsize=11,
        fontweight="bold",
        color="white",
        ha="center",
        va="center",
        zorder=22
    )

    text.set_path_effects([
        pe.withStroke(linewidth=3, foreground="black")
    ])


# ============================================================
# Plot KML layers
# ============================================================

for layer_name, info in study_layers_3857.items():

    # Make the smaller boundary dashed
    linestyle = "--" if layer_name == "East_River" else "-"

    plot_kml_layer(
        ax=ax,
        gdf=info["gdf"],
        color=info["color"],
        linewidth=3.0,
        linestyle=linestyle,
        label=layer_name
    )


# # ============================================================
# # Plot SAIL locations
# # ============================================================

# locations_3857.plot(
#     ax=ax,
#     color=["black", "magenta"],
#     markersize=140,
#     zorder=40
# )


# # ============================================================
# # Add location labels
# # ============================================================

# for idx, row in locations_3857.iterrows():
#     x = row.geometry.x
#     y = row.geometry.y

#     if "SAIL" in row["name"]:
#         label = "SAIL site\nAdjacent Gunnison County Rd 317"
#         color = "black"
#         xytext = (8, 8)
#     else:
#         label = "Field pyranometer\nAdjacent hill"
#         color = "magenta"
#         xytext = (8, -22)

#     text = ax.annotate(
#         label,
#         xy=(x, y),
#         xytext=xytext,
#         textcoords="offset points",
#         fontsize=13,
#         color=color,
#         zorder=41
#     )

#     text.set_path_effects([
#         pe.withStroke(linewidth=3, foreground="white")
#     ])


# ============================================================
# Add a larger regional marker for the East River/SAIL area
# This helps the location stand out on the zoomed-out state map.
# ============================================================

sail_center = locations_3857.unary_union.centroid

ax.scatter(
    sail_center.x,
    sail_center.y,
    s=800,
    marker="*",
    color="yellow",
    edgecolor="black",
    linewidth=1.2,
    zorder=45
)

text = ax.annotate(
    "East River/SAIL area",
    xy=(sail_center.x, sail_center.y),
    xytext=(12, 12),
    textcoords="offset points",
    fontsize=14,
    fontweight="bold",
    color="yellow",
    zorder=46
)

text.set_path_effects([
    pe.withStroke(linewidth=3, foreground="black")
])


# ============================================================
# Final styling
# ============================================================

ax.set_title(
    "East River Study Area in Colorado and Surrounding States",
    fontsize=20
)

# Since the basemap is in Web Mercator, hide projected x/y axes
ax.set_axis_off()


# ============================================================
# Legend
# ============================================================

legend_handles = [
    Line2D([0], [0], color="black", lw=1.2, label="State boundaries"),
    # Line2D([0], [0], color="red", lw=3.0, label="East River Basin"),
    # Line2D([0], [0], color="blue", lw=3.0, linestyle="--", label="East River"),
    # Line2D(
    #     [0], [0],
    #     marker="o",
    #     color="w",
    #     markerfacecolor="black",
    #     markeredgecolor="black",
    #     markersize=9,
    #     label="SAIL site"
    # ),
    # Line2D(
    #     [0], [0],
    #     marker="o",
    #     color="w",
    #     markerfacecolor="magenta",
    #     markeredgecolor="magenta",
    #     markersize=9,
    #     label="Field pyranometer"
    # ),
    Line2D(
        [0], [0],
        marker="*",
        color="w",
        markerfacecolor="yellow",
        markeredgecolor="black",
        markersize=20,
        label="East River/SAIL area"
    )
]

ax.legend(
    handles=legend_handles,
    loc="lower right",
    fontsize=12,
    frameon=True,
    facecolor="white",
    framealpha=0.9
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# KML file path and layer names
# ============================================================

kml_path = "/bsuhome/tnde/geoscience/albedo_downscaling/shape_files/East_River.kml"

kml_layers = {
    "East_River.kml": "red",   # larger boundary
    "East_River": "blue"       # smaller boundary
}

# ============================================================
# Read both KML layers in lon/lat
# ============================================================

study_layers = {}

for layer_name, color in kml_layers.items():

    gdf = gpd.read_file(kml_path, layer=layer_name)

    gdf = gdf[
        gdf.geometry.notnull() & ~gdf.geometry.is_empty
    ].copy()

    # KML is already lon/lat.
    # Only assign CRS if GeoPandas did not detect it.
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326")

    # Keep everything in lon/lat for the regional map
    gdf = gdf.to_crs("EPSG:4326")

    study_layers[layer_name] = {
        "gdf": gdf,
        "color": color
    }

    print(f"\nLayer: {layer_name}")
    print("CRS:", gdf.crs)
    print("Geometry types:", gdf.geometry.geom_type.unique())
    print("Bounds:", gdf.total_bounds)

# ============================================================
# Read UCRB polygon from separate KML file
# ============================================================
ucrb_kml_path = "/bsuhome/tnde/geoscience/albedo_downscaling/shape_files/Colorado_River_Basin_Hydrological_Boundaries.kml"

ucrb_gdf = gpd.read_file(ucrb_kml_path)

ucrb_gdf = ucrb_gdf[
    ucrb_gdf.geometry.notnull() & ~ucrb_gdf.geometry.is_empty
].copy()

if ucrb_gdf.crs is None:
    ucrb_gdf = ucrb_gdf.set_crs("EPSG:4326")

ucrb_gdf = ucrb_gdf.to_crs("EPSG:4326")

# Select polygon 8.
# Python indexing starts from 0, so polygon 8 is index 7.
ucrb_polygon = ucrb_gdf.iloc[[7]].copy()

print("\nSelected UCRB polygon:")
print("CRS:", ucrb_polygon.crs)
print("Geometry types:", ucrb_polygon.geometry.geom_type.unique())
print("Bounds:", ucrb_polygon.total_bounds)

# ============================================================
# SAIL locations in lon/lat
# ============================================================

# 38°57'22.35"N, 106°59'16.66"W
sail_road_lon = -106.987961
sail_road_lat = 38.956208

# 38°57'22.99"N, 106°59'8.79"W
sail_hill_lon = -106.985775
sail_hill_lat = 38.956386

locations_gdf = gpd.GeoDataFrame(
    {
        "name": [
            "SAIL site near County Road 317",
            "Field pyranometer on adjacent hill"
        ],
        "color": [
            "black",
            "magenta"
        ]
    },
    geometry=[
        Point(sail_road_lon, sail_road_lat),
        Point(sail_hill_lon, sail_hill_lat)
    ],
    crs="EPSG:4326"
)


# ============================================================
# Read Colorado and surrounding state boundaries
# ============================================================

states_url = "https://www2.census.gov/geo/tiger/GENZ2023/shp/cb_2023_us_state_500k.zip"

states = gpd.read_file(states_url).to_crs("EPSG:4326")

state_names = [
    "Colorado",
    "Utah",
    "Wyoming",
    "Nebraska",
    "Kansas",
    "Oklahoma",
    "New Mexico",
    "Arizona",
    "Texas"
]

states_region = states[states["NAME"].isin(state_names)].copy()
colorado = states_region[states_region["NAME"] == "Colorado"].copy()


# ============================================================
# Helper functions
# ============================================================

def plot_kml_layer(ax, gdf, color, linewidth=2.5, linestyle="-", zorder=10):
    """
    Plot one KML layer.
    Polygons are plotted as boundaries.
    Lines are plotted as lines.
    Points are plotted as points.
    """

    polygon_types = ["Polygon", "MultiPolygon"]
    line_types = ["LineString", "MultiLineString", "LinearRing"]
    point_types = ["Point", "MultiPoint"]

    poly_gdf = gdf[gdf.geometry.geom_type.isin(polygon_types)]
    if not poly_gdf.empty:
        poly_gdf.boundary.plot(
            ax=ax,
            color=color,
            linewidth=linewidth,
            linestyle=linestyle,
            zorder=zorder
        )

    line_gdf = gdf[gdf.geometry.geom_type.isin(line_types)]
    if not line_gdf.empty:
        line_gdf.plot(
            ax=ax,
            color=color,
            linewidth=linewidth,
            linestyle=linestyle,
            zorder=zorder
        )

    point_gdf = gdf[gdf.geometry.geom_type.isin(point_types)]
    if not point_gdf.empty:
        point_gdf.plot(
            ax=ax,
            color=color,
            markersize=35,
            zorder=zorder
        )

    other_gdf = gdf[
        ~gdf.geometry.geom_type.isin(polygon_types + line_types + point_types)
    ]
    if not other_gdf.empty:
        other_gdf.plot(
            ax=ax,
            facecolor="none",
            edgecolor=color,
            color=color,
            linewidth=linewidth,
            linestyle=linestyle,
            zorder=zorder
        )


def lon_formatter(x, pos):
    return f"{abs(x):.0f}$^\circ$W" if x < 0 else f"{x:.0f}$^\circ$E"
def lat_formatter(y, pos):
    return f"{abs(y):.0f}$^\circ$S" if y < 0 else f"{y:.0f}$^\circ$N"


def lon_formatter_2dp(x, pos):
    return f"{abs(x):.2f}$^\circ$W" if x < 0 else f"{x:.2f}$^\circ$E"
def lat_formatter_2dp(y, pos):
    return f"{abs(y):.2f}$^\circ$S" if y < 0 else f"{y:.2f}$^\circ$N"


# ============================================================
# Create main figure
# ============================================================
fig, ax = plt.subplots(figsize=(11, 10))


# ============================================================
# Plot surrounding states
# ============================================================

states_region.plot(
    ax=ax,
    facecolor="#55a87f",
    edgecolor="black",
    linestyle="-.",
    linewidth=0.8,
    zorder=1
)

# Highlight Colorado slightly
colorado.plot(
    ax=ax,
    facecolor="#4b9c78",
    edgecolor="black",
    linestyle="-.",
    linewidth=1.0,
    zorder=2
)


# ============================================================
# Add state labels
# ============================================================

for _, row in states_region.iterrows():
    point = row.geometry.representative_point()

    text = ax.text(
        point.x,
        point.y,
        row["NAME"],
        fontsize=13,
        color="white",
        ha="center",
        va="center",
        zorder=5
    )

    text.set_path_effects([
        pe.withStroke(linewidth=2.5, foreground="black")
    ])


# ============================================================
# Plot KML layers on main map
# ============================================================

# for layer_name, info in study_layers.items():

#     linestyle = "--" if layer_name == "East_River" else "-"

#     plot_kml_layer(
#         ax=ax,
#         gdf=info["gdf"],
#         color=info["color"],
#         linewidth=2.2,
#         linestyle=linestyle,
#         zorder=20
#     )


# ============================================================
# Plot UCRB polygon on main map only
# ============================================================

plot_kml_layer(
    ax=ax,
    gdf=ucrb_polygon,
    color="purple",
    linewidth=2.5,
    linestyle="--",
    zorder=20
)

# # ============================================================
# # Plot East River boundary on main map
# # ============================================================

# if "East_River" in study_layers:

#     plot_kml_layer(
#         ax=ax,
#         gdf=study_layers["East_River"]["gdf"],
#         color="blue",
#         linewidth=2.2,
#         linestyle="--",
#         zorder=21
#     )

# # ============================================================
# # Plot SAIL locations on main map
# # ============================================================

# locations_gdf.plot(
#     ax=ax,
#     color=["black", "magenta"],
#     markersize=55,
#     zorder=25
# )

# Add a larger star so the area is visible at state scale
sail_center = locations_gdf.geometry.unary_union.centroid

# ax.scatter(
#     sail_center.x,
#     sail_center.y,
#     s=750,
#     marker="*",
#     color="yellow",
#     edgecolor="black",
#     linewidth=1.0,
#     zorder=30
# )

text = ax.annotate(
    "ERB",
    xy=(sail_center.x, sail_center.y),
    xytext=(-13, 8),
    textcoords="offset points",
    fontsize=12,
    fontweight="bold",
    color="white",
    zorder=31
)

text.set_path_effects([
    pe.withStroke(linewidth=2.5, foreground="black")
])


# ============================================================
# Set main map extent
# ============================================================

xmin, ymin, xmax, ymax = states_region.total_bounds

xpad = (xmax - xmin) * 0.05
ypad = (ymax - ymin) * 0.05

ax.set_xlim(xmin - xpad, xmax + xpad)
ax.set_ylim(ymin - ypad, ymax + ypad)


# ============================================================
# Add zoomed inset for the KML boundaries
# ============================================================

# axins = inset_axes(
#     ax,
#     width="38%",
#     height="38%",
#     loc="upper right",
#     borderpad=2.50
# )

axins = inset_axes(
    ax,
    width="47%",
    height="43%",
    loc="upper right",
    bbox_to_anchor=(0.068, 0.0, 1, 1),  # shift inset slightly to the right
    bbox_transform=ax.transAxes,
    borderpad=2.0
)

# Light background for inset
axins.set_facecolor("#f5f5f5")

# Plot the KML layers in the inset
for layer_name, info in study_layers.items():

    linestyle = ":" if layer_name == "East_River" else "-"

    plot_kml_layer(
        ax=axins,
        gdf=info["gdf"],
        color=info["color"],
        linewidth=2.5,
        linestyle=linestyle,
        zorder=20
    )

# Plot SAIL locations in the inset
# locations_gdf.plot(
#     ax=axins,
#     facecolor="none",
#     color=["black", "magenta"],
#     linewidth=3.50,
#     markersize=65,
#     zorder=25
# )

# Black location as a circle
locations_gdf[locations_gdf["color"] == "black"].plot(
    ax=axins,
    marker="o",
    facecolor="black",
    edgecolor="black",
    linewidth=3.50,
    markersize=85,
    zorder=25
)

# Magenta location as a star
locations_gdf[locations_gdf["color"] == "magenta"].plot(
    ax=axins,
    marker="*",
    facecolor="magenta",
    edgecolor="magenta",
    linewidth=1.0,
    markersize=120,
    zorder=26
)


# Inset extent based on KML and site locations
all_bounds = []

for info in study_layers.values():
    all_bounds.append(info["gdf"].total_bounds)

all_bounds.append(locations_gdf.total_bounds)

ixmin = min(b[0] for b in all_bounds)
iymin = min(b[1] for b in all_bounds)
ixmax = max(b[2] for b in all_bounds)
iymax = max(b[3] for b in all_bounds)

ixpad = (ixmax - ixmin) * 0.12
iypad = (iymax - iymin) * 0.12

axins.set_xlim(ixmin - ixpad, ixmax + ixpad)
axins.set_ylim(iymin - iypad, iymax + iypad)

###################################################
###################################################
# axins.set_title("UCRB and East River Boundaries", fontsize=11)
# axins.tick_params(axis="both", labelsize=8)
# axins.xaxis.set_major_formatter(FuncFormatter(lon_formatter))
# axins.yaxis.set_major_formatter(FuncFormatter(lat_formatter))
# axins.grid(True, alpha=0.3)


axins.set_title("East River Study Area with SAIL Location", fontsize=11, fontweight="bold")
# axins.set_xlabel("Longitude", fontsize=9)
# axins.set_ylabel("Latitude", fontsize=9)
axins.tick_params(axis="both", labelsize=8)
axins.xaxis.set_major_formatter(FuncFormatter(lon_formatter_2dp))
axins.yaxis.set_major_formatter(FuncFormatter(lat_formatter_2dp))

# Make the grid spacing larger
axins.xaxis.set_major_locator(MultipleLocator(0.06))
axins.yaxis.set_major_locator(MultipleLocator(0.06))

# Make grid lines more visible
axins.grid(
    True,
    alpha=0.6,
    linewidth=1.0,
    linestyle="--"
)
# axins.grid(True, alpha=0.3)
###################################################
###################################################

# Draw a rectangle on main map showing the inset area
rect = Rectangle(
    (ixmin - ixpad, iymin - iypad),
    (ixmax - ixmin) + 2 * ixpad,
    (iymax - iymin) + 2 * iypad,
    linewidth=3.0,
    edgecolor="maroon",
    facecolor="none",
    linestyle="-",
    zorder=26
)

ax.add_patch(rect)


# ============================================================
# Add north arrow to the zoomed inset
# ============================================================

axins.annotate(
    "",
    xy=(0.89, 0.87),       # arrow head
    xytext=(0.89, 0.69),   # arrow tail
    xycoords="axes fraction",
    arrowprops=dict(
        facecolor="black",
        edgecolor="black",
        width=6,
        headwidth=18,
        headlength=35
    ),
    zorder=50
)

axins.text(
    0.89, 0.89,
    "N",
    transform=axins.transAxes,
    ha="center",
    va="bottom",
    fontsize=25,
    fontweight="bold",
    color="black",
    zorder=51,
    bbox=dict(facecolor="white", edgecolor="none", alpha=0.8, pad=1.5)
)


# ============================================================
# Add scale bar to the zoomed inset
# ============================================================

# Choose scale bar length in km
scale_km = 2.0

# Mid-latitude of inset, used to convert longitude degrees to km
mid_lat = 0.5 * ((iymin - iypad) + (iymax + iypad))

# Approximate km per degree longitude at this latitude
km_per_deg_lon = 111.32 * math.cos(math.radians(mid_lat))

# Convert desired scale length from km to degrees longitude
scale_deg = scale_km / km_per_deg_lon

# Position the scale bar near the lower-left of the inset
x0 = (ixmin - ixpad) + 0.08 * ((ixmax + ixpad) - (ixmin - ixpad))
y0 = (iymin - iypad) + 0.08 * ((iymax + iypad) - (iymin - iypad))

# Draw horizontal scale bar
axins.plot(
    [x0, x0 + scale_deg],
    [y0, y0],
    color="black",
    linewidth=3,
    zorder=50
)

# End ticks
tick_h = 0.015 * ((iymax + iypad) - (iymin - iypad))

axins.plot(
    [x0, x0],
    [y0 - tick_h, y0 + tick_h],
    color="black",
    linewidth=3,
    zorder=50
)

axins.plot(
    [x0 + scale_deg, x0 + scale_deg],
    [y0 - tick_h, y0 + tick_h],
    color="black",
    linewidth=3,
    zorder=50
)

# Scale label
axins.text(
    x0 + scale_deg / 2,
    y0 + 1.8 * tick_h,
    f"{scale_km:.0f} km",
    ha="center",
    va="bottom",
    fontsize=11, 
    fontweight="bold",
    color="black",
    zorder=51,
    bbox=dict(facecolor="white", edgecolor="none", alpha=0.8, pad=1.2)
)


# ============================================================
# Add north arrow to the main plot
# ============================================================

ax.annotate(
    "",
    xy=(0.92, 0.08),       # arrow head
    xytext=(0.92, 0.01),   # arrow tail
    xycoords="axes fraction",
    arrowprops=dict(
        facecolor="black",
        edgecolor="black",
        width=6,
        headwidth=20,
        headlength=35
    ),
    zorder=50
)

ax.text(
    0.92, 0.08,
    "N",
    transform=ax.transAxes,
    ha="center",
    va="bottom",
    fontsize=30,
    fontweight="bold",
    color="black",
    zorder=51,
    bbox=dict(facecolor="white", edgecolor="none", alpha=0.8, pad=1.5)
)

# ============================================================
# Axes formatting
# ============================================================

ax.set_title(
    "Upper Colorado River Basin (UCRB) within Colorado and Surrounding States",
    fontsize=16, fontweight="bold"
)

ax.set_xlabel("Longitude", fontsize=13)
ax.set_ylabel("Latitude", fontsize=13)
ax.xaxis.set_major_formatter(FuncFormatter(lon_formatter))
ax.yaxis.set_major_formatter(FuncFormatter(lat_formatter))
ax.tick_params(axis="both", labelsize=11)
ax.grid(True, alpha=0.25)
ax.set_aspect("equal")



# ============================================================
# Legend
# ============================================================
legend_handles = [
    Line2D([0], [0], color="black", lw=1.2, linestyle="-.", label="State boundary"),
    Line2D([0], [0], color="purple", lw=2.5, linestyle="--", label="UCRB boundary"),
    Line2D([0], [0], color="red", lw=2.5, label="ERB boundary"),
    Line2D([0], [0], color="blue", lw=2.5, linestyle=":", label="East River Watershed boundary"),
    Line2D(
        [0], [0],
        marker="s",
        color="w",
        markerfacecolor="none",
        markeredgecolor="maroon",
        markeredgewidth=3.30,
        markersize=10,
        label="ERB location"
    ),
    Line2D(
        [0], [0],
        marker="o",
        color="w",
        markerfacecolor="black",
        markeredgecolor="black",
        markeredgewidth=3.30,
        markersize=8,
        label="SAIL site"
    ),
    Line2D(
        [0], [0],
        marker="*",
        color="w",
        markerfacecolor="magenta",
        markeredgecolor="magenta",
        markeredgewidth=1.0,
        markersize=15,
        label="Field pyranometer"
    )
]

ax.legend(
    handles=legend_handles,
    loc="lower left",
    fontsize=10,
    frameon=True,
    facecolor="white",
    framealpha=1.0
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# KML file path and layer names
# ============================================================

kml_path = "/bsuhome/tnde/geoscience/albedo_downscaling/shape_files/East_River.kml"

kml_layers = {
    "East_River.kml": "red",   # larger boundary
    "East_River": "blue"       # smaller boundary
}

# ============================================================
# Read both KML layers in lon/lat
# ============================================================

study_layers = {}

for layer_name, color in kml_layers.items():

    gdf = gpd.read_file(kml_path, layer=layer_name)

    gdf = gdf[
        gdf.geometry.notnull() & ~gdf.geometry.is_empty
    ].copy()

    # KML is already lon/lat.
    # Only assign CRS if GeoPandas did not detect it.
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326")

    # Keep everything in lon/lat for the regional map
    gdf = gdf.to_crs("EPSG:4326")

    study_layers[layer_name] = {
        "gdf": gdf,
        "color": color
    }

    print(f"\nLayer: {layer_name}")
    print("CRS:", gdf.crs)
    print("Geometry types:", gdf.geometry.geom_type.unique())
    print("Bounds:", gdf.total_bounds)

# ============================================================
# Read UCRB polygon from separate KML file
# ============================================================
ucrb_kml_path = "/bsuhome/tnde/geoscience/albedo_downscaling/shape_files/Colorado_River_Basin_Hydrological_Boundaries.kml"

ucrb_gdf = gpd.read_file(ucrb_kml_path)

ucrb_gdf = ucrb_gdf[
    ucrb_gdf.geometry.notnull() & ~ucrb_gdf.geometry.is_empty
].copy()

if ucrb_gdf.crs is None:
    ucrb_gdf = ucrb_gdf.set_crs("EPSG:4326")

ucrb_gdf = ucrb_gdf.to_crs("EPSG:4326")

# Select polygon 8.
# Python indexing starts from 0, so polygon 8 is index 7.
ucrb_polygon = ucrb_gdf.iloc[[7]].copy()

print("\nSelected UCRB polygon:")
print("CRS:", ucrb_polygon.crs)
print("Geometry types:", ucrb_polygon.geometry.geom_type.unique())
print("Bounds:", ucrb_polygon.total_bounds)    

# ============================================================
# SAIL locations in lon/lat
# ============================================================

# 38°57'22.35"N, 106°59'16.66"W
sail_road_lon = -106.987961
sail_road_lat = 38.956208

# 38°57'22.99"N, 106°59'8.79"W
sail_hill_lon = -106.985775
sail_hill_lat = 38.956386

locations_gdf = gpd.GeoDataFrame(
    {
        "name": [
            "SAIL site near County Road 317",
            "Field pyranometer on adjacent hill"
        ],
        "color": [
            "black",
            "magenta"
        ]
    },
    geometry=[
        Point(sail_road_lon, sail_road_lat),
        Point(sail_hill_lon, sail_hill_lat)
    ],
    crs="EPSG:4326"
)

# ============================================================
# Read Colorado and surrounding state boundaries
# ============================================================

states_url = "https://www2.census.gov/geo/tiger/GENZ2023/shp/cb_2023_us_state_500k.zip"

states = gpd.read_file(states_url).to_crs("EPSG:4326")

state_names = [
    "Colorado",
    "Utah",
    "Wyoming",
    "Nebraska",
    "Kansas",
    "Oklahoma",
    "New Mexico",
    "Arizona",
    "Texas"
]

states_region = states[states["NAME"].isin(state_names)].copy()
colorado = states_region[states_region["NAME"] == "Colorado"].copy()


# ============================================================
# Helper functions
# ============================================================

def plot_kml_layer(ax, gdf, color, linewidth=2.5, linestyle="-", zorder=10):
    """
    Plot one KML layer.
    Polygons are plotted as boundaries.
    Lines are plotted as lines.
    Points are plotted as points.
    """

    polygon_types = ["Polygon", "MultiPolygon"]
    line_types = ["LineString", "MultiLineString", "LinearRing"]
    point_types = ["Point", "MultiPoint"]

    poly_gdf = gdf[gdf.geometry.geom_type.isin(polygon_types)]
    if not poly_gdf.empty:
        poly_gdf.boundary.plot(
            ax=ax,
            color=color,
            linewidth=linewidth,
            linestyle=linestyle,
            zorder=zorder
        )

    line_gdf = gdf[gdf.geometry.geom_type.isin(line_types)]
    if not line_gdf.empty:
        line_gdf.plot(
            ax=ax,
            color=color,
            linewidth=linewidth,
            linestyle=linestyle,
            zorder=zorder
        )

    point_gdf = gdf[gdf.geometry.geom_type.isin(point_types)]
    if not point_gdf.empty:
        point_gdf.plot(
            ax=ax,
            color=color,
            markersize=35,
            zorder=zorder
        )

    other_gdf = gdf[
        ~gdf.geometry.geom_type.isin(polygon_types + line_types + point_types)
    ]
    if not other_gdf.empty:
        other_gdf.plot(
            ax=ax,
            facecolor="none",
            edgecolor=color,
            color=color,
            linewidth=linewidth,
            linestyle=linestyle,
            zorder=zorder
        )


def lon_formatter(x, pos):
    return f"{abs(x):.0f}$^\circ$W" if x < 0 else f"{x:.0f}$^\circ$E"
def lat_formatter(y, pos):
    return f"{abs(y):.0f}$^\circ$S" if y < 0 else f"{y:.0f}$^\circ$N"


def lon_formatter_2dp(x, pos):
    return f"{abs(x):.2f}$^\circ$W" if x < 0 else f"{x:.2f}$^\circ$E"
def lat_formatter_2dp(y, pos):
    return f"{abs(y):.2f}$^\circ$S" if y < 0 else f"{y:.2f}$^\circ$N"

# Add 2-decimal tick formatters for the inset in Web Mercator
merc_to_geo = Transformer.from_crs("EPSG:3857", "EPSG:4326", always_xy=True)
def lon_formatter_2dp_3857(x, pos):
    lon, _ = merc_to_geo.transform(x, 0)
    return f"{abs(lon):.2f}$^\circ$W" if lon < 0 else f"{lon:.2f}$^\circ$E"
def lat_formatter_2dp_3857(y, pos):
    _, lat = merc_to_geo.transform(0, y)
    return f"{abs(lat):.2f}$^\circ$S" if lat < 0 else f"{lat:.2f}$^\circ$N"


# ============================================================
# Create main figure
# ============================================================
fig, ax = plt.subplots(figsize=(11, 10))


# ============================================================
# Plot surrounding states
# ============================================================

states_region.plot(
    ax=ax,
    facecolor="#55a87f",
    edgecolor="black",
    linestyle="-.",
    linewidth=0.8,
    zorder=1
)

# Highlight Colorado slightly
colorado.plot(
    ax=ax,
    facecolor="#4b9c78",
    edgecolor="black",
    linestyle="-.",
    linewidth=1.0,
    zorder=2
)

# ============================================================
# Add state labels
# ============================================================

for _, row in states_region.iterrows():
    point = row.geometry.representative_point()

    text = ax.text(
        point.x,
        point.y,
        row["NAME"],
        fontsize=13,
        color="white",
        ha="center",
        va="center",
        zorder=5
    )

    text.set_path_effects([
        pe.withStroke(linewidth=2.5, foreground="black")
    ])


# # ============================================================
# # Plot KML layers on main map
# # ============================================================

# for layer_name, info in study_layers.items():

#     linestyle = "--" if layer_name == "East_River" else "-"

#     plot_kml_layer(
#         ax=ax,
#         gdf=info["gdf"],
#         color=info["color"],
#         linewidth=2.2,
#         linestyle=linestyle,
#         zorder=20
#     )

# ============================================================
# Plot UCRB polygon on main map only
# ============================================================

plot_kml_layer(
    ax=ax,
    gdf=ucrb_polygon,
    color="purple",
    linewidth=2.5,
    linestyle="--",
    zorder=20
)

# # ============================================================
# # Plot East River boundary on main map
# # ============================================================

# if "East_River" in study_layers:

#     plot_kml_layer(
#         ax=ax,
#         gdf=study_layers["East_River"]["gdf"],
#         color="blue",
#         linewidth=2.2,
#         linestyle="--",
#         zorder=21
#     )

# # ============================================================
# # Plot SAIL locations on main map
# # ============================================================

# locations_gdf.plot(
#     ax=ax,
#     color=["blue", "magenta"],
#     markersize=55,
#     zorder=25
# )

# Add a larger star so the area is visible at state scale
sail_center = locations_gdf.geometry.unary_union.centroid

# ax.scatter(
#     sail_center.x,
#     sail_center.y,
#     s=750,
#     marker="*",
#     color="yellow",
#     edgecolor="black",
#     linewidth=1.0,
#     zorder=30
# )

text = ax.annotate(
    "ERB",
    xy=(sail_center.x, sail_center.y),
    xytext=(-13, 8),
    textcoords="offset points",
    fontsize=12,
    fontweight="bold",
    color="white",
    zorder=31
)

text.set_path_effects([
    pe.withStroke(linewidth=2.5, foreground="black")
])


# ============================================================
# Set main map extent
# ============================================================

xmin, ymin, xmax, ymax = states_region.total_bounds

xpad = (xmax - xmin) * 0.05
ypad = (ymax - ymin) * 0.05

ax.set_xlim(xmin - xpad, xmax + xpad)
ax.set_ylim(ymin - ypad, ymax + ypad)


# ============================================================
# Add zoomed inset for the KML boundaries with Esri World Imagery
# ============================================================

# axins = inset_axes(
#     ax,
#     width="38%",
#     height="38%",
#     loc="upper right",
#     borderpad=2.50
# )

axins = inset_axes(
    ax,
    width="47%",
    height="43%",
    loc="upper right",
    bbox_to_anchor=(0.068, 0.0, 1, 1),  # shift inset slightly to the right
    bbox_transform=ax.transAxes,
    borderpad=2.0
)

# Light background for inset
axins.set_facecolor("#f5f5f5")

# Inset extent based on KML and site locations
all_bounds = []

for info in study_layers.values():
    all_bounds.append(info["gdf"].total_bounds)

all_bounds.append(locations_gdf.total_bounds)

ixmin = min(b[0] for b in all_bounds)
iymin = min(b[1] for b in all_bounds)
ixmax = max(b[2] for b in all_bounds)
iymax = max(b[3] for b in all_bounds)

ixpad = (ixmax - ixmin) * 0.12
iypad = (iymax - iymin) * 0.12

axins.set_xlim(ixmin - ixpad, ixmax + ixpad)
axins.set_ylim(iymin - iypad, iymax + iypad)

# ============================================================
# Add Esri World Imagery to the zoomed inset only
# ============================================================

ctx.add_basemap(
    axins,
    source=ctx.providers.Esri.WorldImagery,
    crs="EPSG:4326",
    zoom=13,
    attribution_size=5
)

# Keep inset limits unchanged after adding basemap
axins.set_xlim(ixmin - ixpad, ixmax + ixpad)
axins.set_ylim(iymin - iypad, iymax + iypad)

# Plot the KML layers in the inset
for layer_name, info in study_layers.items():

    linestyle = ":" if layer_name == "East_River" else "-"

    plot_kml_layer(
        ax=axins,
        gdf=info["gdf"],
        color=info["color"],
        linewidth=2.5,
        linestyle=linestyle,
        zorder=20
    )

# Plot SAIL locations in the inset

# locations_gdf.plot(
#     ax=axins,
#     facecolor="none",
#     color=["blue", "magenta"],
#     linewidth=3.50,
#     markersize=65,
#     zorder=25
# )

# Black location as a circle
locations_gdf[locations_gdf["color"] == "black"].plot(
    ax=axins,
    marker="o",
    facecolor="blue",
    edgecolor="blue",
    linewidth=3.50,
    markersize=85,
    zorder=25
)

# Magenta location as a star
locations_gdf[locations_gdf["color"] == "magenta"].plot(
    ax=axins,
    marker="*",
    facecolor="magenta",
    edgecolor="magenta",
    linewidth=1.0,
    markersize=120,
    zorder=26
)

###################################################
###################################################
# axins.set_title("UCRB and East River Boundaries", fontsize=11)
# axins.tick_params(axis="both", labelsize=8)
# axins.xaxis.set_major_formatter(FuncFormatter(lon_formatter))
# axins.yaxis.set_major_formatter(FuncFormatter(lat_formatter))
# axins.grid(True, alpha=0.3)


axins.set_title("East River Study Area with SAIL Location", fontsize=11, fontweight="bold")
# axins.set_xlabel("Longitude", fontsize=9)
# axins.set_ylabel("Latitude", fontsize=9)
axins.tick_params(axis="both", labelsize=8)
axins.xaxis.set_major_formatter(FuncFormatter(lon_formatter_2dp))
axins.yaxis.set_major_formatter(FuncFormatter(lat_formatter_2dp))

# Make the grid spacing larger
axins.xaxis.set_major_locator(MultipleLocator(0.06))
axins.yaxis.set_major_locator(MultipleLocator(0.06))

# Make grid lines more visible
axins.grid(
    True,
    alpha=0.6,
    linewidth=1.0,
    linestyle="--"
)
# axins.grid(True, alpha=0.3)
###################################################
###################################################

# Draw a rectangle on main map showing the inset area
rect = Rectangle(
    (ixmin - ixpad, iymin - iypad),
    (ixmax - ixmin) + 2 * ixpad,
    (iymax - iymin) + 2 * iypad,
    linewidth=3.0,
    edgecolor="maroon",
    facecolor="none",
    linestyle="-",
    zorder=26
)

ax.add_patch(rect)


# ============================================================
# Add north arrow to the zoomed inset
# ============================================================

axins.annotate(
    "",
    xy=(0.89, 0.87),       # arrow head
    xytext=(0.89, 0.69),   # arrow tail
    xycoords="axes fraction",
    arrowprops=dict(
        facecolor="black",
        edgecolor="black",
        width=6,
        headwidth=18,
        headlength=35
    ),
    zorder=50
)

axins.text(
    0.89, 0.89,
    "N",
    transform=axins.transAxes,
    ha="center",
    va="bottom",
    fontsize=25,
    fontweight="bold",
    color="black",
    zorder=51,
    bbox=dict(facecolor="white", edgecolor="none", alpha=0.8, pad=1.5)
)


# ============================================================
# Add scale bar to the zoomed inset
# ============================================================

# Choose scale bar length in km
scale_km = 2.0

# Mid-latitude of inset, used to convert longitude degrees to km
mid_lat = 0.5 * ((iymin - iypad) + (iymax + iypad))

# Approximate km per degree longitude at this latitude
km_per_deg_lon = 111.32 * math.cos(math.radians(mid_lat))

# Convert desired scale length from km to degrees longitude
scale_deg = scale_km / km_per_deg_lon

# Position the scale bar near the lower-left of the inset
x0 = (ixmin - ixpad) + 0.08 * ((ixmax + ixpad) - (ixmin - ixpad))
y0 = (iymin - iypad) + 0.08 * ((iymax + iypad) - (iymin - iypad))

# Draw horizontal scale bar
axins.plot(
    [x0, x0 + scale_deg],
    [y0, y0],
    color="black",
    linewidth=3,
    zorder=50
)

# End ticks
tick_h = 0.015 * ((iymax + iypad) - (iymin - iypad))

axins.plot(
    [x0, x0],
    [y0 - tick_h, y0 + tick_h],
    color="black",
    linewidth=3,
    zorder=50
)

axins.plot(
    [x0 + scale_deg, x0 + scale_deg],
    [y0 - tick_h, y0 + tick_h],
    color="black",
    linewidth=3,
    zorder=50
)

# Scale label
axins.text(
    x0 + scale_deg / 2,
    y0 + 1.8 * tick_h,
    f"{scale_km:.0f} km",
    ha="center",
    va="bottom",
    fontsize=11,
    fontweight="bold",
    color="black",
    zorder=51,
    bbox=dict(facecolor="white", edgecolor="none", alpha=0.8, pad=1.2)
)


# ============================================================
# Add north arrow to the main plot
# ============================================================

ax.annotate(
    "",
    xy=(0.92, 0.08),       # arrow head
    xytext=(0.92, 0.01),   # arrow tail
    xycoords="axes fraction",
    arrowprops=dict(
        facecolor="black",
        edgecolor="black",
        width=6,
        headwidth=20,
        headlength=35
    ),
    zorder=50
)

ax.text(
    0.92, 0.08,
    "N",
    transform=ax.transAxes,
    ha="center",
    va="bottom",
    fontsize=30,
    fontweight="bold",
    color="black",
    zorder=51,
    bbox=dict(facecolor="white", edgecolor="none", alpha=0.8, pad=1.5)
)

# ============================================================
# Axes formatting
# ============================================================

ax.set_title(
    "Upper Colorado River Basin (UCRB) within Colorado and Surrounding States",
    fontsize=16, fontweight="bold"
)

ax.set_xlabel("Longitude", fontsize=13)
ax.set_ylabel("Latitude", fontsize=13)
ax.xaxis.set_major_formatter(FuncFormatter(lon_formatter))
ax.yaxis.set_major_formatter(FuncFormatter(lat_formatter))
ax.tick_params(axis="both", labelsize=11)
ax.grid(True, alpha=0.25)
ax.set_aspect("equal")


# ============================================================
# Legend
# ============================================================
legend_handles = [
    Line2D([0], [0], color="black", lw=1.2, linestyle="-.", label="State boundary"),
    Line2D([0], [0], color="purple", lw=2.5, linestyle="--", label="UCRB boundary"),
    Line2D([0], [0], color="red", lw=2.5, label="ERB boundary"),
    Line2D([0], [0], color="blue", lw=2.5, linestyle=":", label="East River Watershed boundary"),
    Line2D(
        [0], [0],
        marker="s",
        color="w",
        markerfacecolor="none",
        markeredgecolor="maroon",
        markeredgewidth=3.30,
        markersize=10,
        label="ERB location"
    ),
    Line2D(
        [0], [0],
        marker="o",
        color="w",
        markerfacecolor="blue",
        markeredgecolor="blue",
        markeredgewidth=3.30,
        markersize=8,
        label="SAIL site"
    ),
    Line2D(
        [0], [0],
        marker="*",
        color="w",
        markerfacecolor="magenta",
        markeredgecolor="magenta",
        markeredgewidth=1.0,
        markersize=15,
        label="Field pyranometer"
    )
]

ax.legend(
    handles=legend_handles,
    loc="lower left",
    fontsize=10,
    frameon=True,
    facecolor="white",
    framealpha=1.0
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# KML file path and layer names
# ============================================================
kml_path = "/bsuhome/tnde/geoscience/albedo_downscaling/shape_files/East_River.kml"

kml_layers = {
    "East_River.kml": "red",   # larger boundary
    "East_River": "blue"       # smaller boundary
}

# ============================================================
# Read both KML layers in lon/lat
# ============================================================
study_layers = {}

for layer_name, color in kml_layers.items():

    gdf = gpd.read_file(kml_path, layer=layer_name)

    gdf = gdf[
        gdf.geometry.notnull() & ~gdf.geometry.is_empty
    ].copy()

    # KML is already lon/lat.
    # Only assign CRS if GeoPandas did not detect it.
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326")

    # Keep everything in lon/lat for the regional map
    gdf = gdf.to_crs("EPSG:4326")

    study_layers[layer_name] = {
        "gdf": gdf,
        "color": color
    }

    print(f"\nLayer: {layer_name}")
    print("CRS:", gdf.crs)
    print("Geometry types:", gdf.geometry.geom_type.unique())
    print("Bounds:", gdf.total_bounds)

# ============================================================
# Read UCRB polygon from separate KML file
# ============================================================
ucrb_kml_path = "/bsuhome/tnde/geoscience/albedo_downscaling/shape_files/Colorado_River_Basin_Hydrological_Boundaries.kml"

ucrb_gdf = gpd.read_file(ucrb_kml_path)

ucrb_gdf = ucrb_gdf[
    ucrb_gdf.geometry.notnull() & ~ucrb_gdf.geometry.is_empty
].copy()

if ucrb_gdf.crs is None:
    ucrb_gdf = ucrb_gdf.set_crs("EPSG:4326")

ucrb_gdf = ucrb_gdf.to_crs("EPSG:4326")

# Select polygon 8.
# Python indexing starts from 0, so polygon 8 is index 7.
ucrb_polygon = ucrb_gdf.iloc[[7]].copy()

print("\nSelected UCRB polygon:")
print("CRS:", ucrb_polygon.crs)
print("Geometry types:", ucrb_polygon.geometry.geom_type.unique())
print("Bounds:", ucrb_polygon.total_bounds)
    
# ============================================================
# SAIL locations in lon/lat
# ============================================================

# 38°57'22.35"N, 106°59'16.66"W
sail_road_lon = -106.987961
sail_road_lat = 38.956208

# 38°57'22.99"N, 106°59'8.79"W
sail_hill_lon = -106.985775
sail_hill_lat = 38.956386

locations_gdf = gpd.GeoDataFrame(
    {
        "name": [
            "SAIL site near County Road 317",
            "Field pyranometer on adjacent hill"
        ],
        "color": [
            "black",
            "magenta"
        ]
    },
    geometry=[
        Point(sail_road_lon, sail_road_lat),
        Point(sail_hill_lon, sail_hill_lat)
    ],
    crs="EPSG:4326"
)


# ============================================================
# Read Colorado and surrounding state boundaries
# ============================================================

states_url = "https://www2.census.gov/geo/tiger/GENZ2023/shp/cb_2023_us_state_500k.zip"

states = gpd.read_file(states_url).to_crs("EPSG:4326")

state_names = [
    "Colorado",
    "Utah",
    "Wyoming",
    "Nebraska",
    "Kansas",
    "Oklahoma",
    "New Mexico",
    "Arizona",
    "Texas"
]

states_region = states[states["NAME"].isin(state_names)].copy()
colorado = states_region[states_region["NAME"] == "Colorado"].copy()


# ============================================================
# Helper functions
# ============================================================

def plot_kml_layer(ax, gdf, color, linewidth=2.5, linestyle="-", zorder=10):
    """
    Plot one KML layer.
    Polygons are plotted as boundaries.
    Lines are plotted as lines.
    Points are plotted as points.
    """

    polygon_types = ["Polygon", "MultiPolygon"]
    line_types = ["LineString", "MultiLineString", "LinearRing"]
    point_types = ["Point", "MultiPoint"]

    poly_gdf = gdf[gdf.geometry.geom_type.isin(polygon_types)]
    if not poly_gdf.empty:
        poly_gdf.boundary.plot(
            ax=ax,
            color=color,
            linewidth=linewidth,
            linestyle=linestyle,
            zorder=zorder
        )

    line_gdf = gdf[gdf.geometry.geom_type.isin(line_types)]
    if not line_gdf.empty:
        line_gdf.plot(
            ax=ax,
            color=color,
            linewidth=linewidth,
            linestyle=linestyle,
            zorder=zorder
        )

    point_gdf = gdf[gdf.geometry.geom_type.isin(point_types)]
    if not point_gdf.empty:
        point_gdf.plot(
            ax=ax,
            color=color,
            markersize=35,
            zorder=zorder
        )

    other_gdf = gdf[
        ~gdf.geometry.geom_type.isin(polygon_types + line_types + point_types)
    ]
    if not other_gdf.empty:
        other_gdf.plot(
            ax=ax,
            facecolor="none",
            edgecolor=color,
            color=color,
            linewidth=linewidth,
            linestyle=linestyle,
            zorder=zorder
        )


def lon_formatter(x, pos):
    return f"{abs(x):.0f}$^\circ$W" if x < 0 else f"{x:.0f}$^\circ$E"
def lat_formatter(y, pos):
    return f"{abs(y):.0f}$^\circ$S" if y < 0 else f"{y:.0f}$^\circ$N"


def lon_formatter_2dp(x, pos):
    return f"{abs(x):.2f}$^\circ$W" if x < 0 else f"{x:.2f}$^\circ$E"
def lat_formatter_2dp(y, pos):
    return f"{abs(y):.2f}$^\circ$S" if y < 0 else f"{y:.2f}$^\circ$N"


# ============================================================
# Create main figure
# ============================================================

fig, ax = plt.subplots(figsize=(11, 10))


# ============================================================
# Plot surrounding states
# ============================================================

states_region.plot(
    ax=ax,
    facecolor="#55a87f",
    edgecolor="black",
    linestyle="-.",
    linewidth=0.8,
    zorder=1
)

# Highlight Colorado slightly
colorado.plot(
    ax=ax,
    facecolor="#4b9c78",
    edgecolor="black",
    linestyle="-.",
    linewidth=1.0,
    zorder=2
)

# ============================================================
# Add state labels
# ============================================================

for _, row in states_region.iterrows():
    point = row.geometry.representative_point()

    text = ax.text(
        point.x,
        point.y,
        row["NAME"],
        fontsize=13,
        color="white",
        ha="center",
        va="center",
        zorder=5
    )

    text.set_path_effects([
        pe.withStroke(linewidth=2.5, foreground="black")
    ])


# # ============================================================
# # Plot KML layers on main map
# # ============================================================

# for layer_name, info in study_layers.items():

#     linestyle = "--" if layer_name == "East_River" else "-"

#     plot_kml_layer(
#         ax=ax,
#         gdf=info["gdf"],
#         color=info["color"],
#         linewidth=2.2,
#         linestyle=linestyle,
#         zorder=20
#     )

# ============================================================
# Plot UCRB polygon on main map only
# ============================================================
plot_kml_layer(
    ax=ax,
    gdf=ucrb_polygon,
    color="purple",
    linewidth=2.5,
    linestyle="--",
    zorder=20
)

# # ============================================================
# # Plot East River boundary on main map
# # ============================================================
# if "East_River" in study_layers:

#     plot_kml_layer(
#         ax=ax,
#         gdf=study_layers["East_River"]["gdf"],
#         color="blue",
#         linewidth=2.2,
#         linestyle="--",
#         zorder=21
#     )

# # ============================================================
# # Plot SAIL locations on main map
# # ============================================================
# locations_gdf.plot(
#     ax=ax,
#     color=["black", "magenta"],
#     markersize=55,
#     zorder=25
# )

# Add a larger star so the area is visible at state scale
sail_center = locations_gdf.geometry.unary_union.centroid

# ax.scatter(
#     sail_center.x,
#     sail_center.y,
#     s=650,
#     marker="*",
#     color="yellow",
#     edgecolor="black",
#     linewidth=1.0,
#     zorder=30
# )


text = ax.annotate(
    "ERB",
    xy=(sail_center.x, sail_center.y),
    xytext=(-13, 8),
    textcoords="offset points",
    fontsize=12,
    fontweight="bold",
    color="white",
    zorder=31
)

text.set_path_effects([
    pe.withStroke(linewidth=2.5, foreground="black")
])


# ============================================================
# Set main map extent
# ============================================================

xmin, ymin, xmax, ymax = states_region.total_bounds

xpad = (xmax - xmin) * 0.05
ypad = (ymax - ymin) * 0.05

ax.set_xlim(xmin - xpad, xmax + xpad)
ax.set_ylim(ymin - ypad, ymax + ypad)


# ============================================================
# Add zoomed inset for the KML boundaries
# ============================================================

# axins = inset_axes(
#     ax,
#     width="38%",
#     height="38%",
#     loc="upper right",
#     borderpad=2.50
# )

axins = inset_axes(
    ax,
    width="47%",
    height="43%",
    loc="upper right",
    bbox_to_anchor=(0.068, 0.0, 1, 1),  # shift inset slightly to the right
    bbox_transform=ax.transAxes,
    borderpad=2.0
)

# Light background for inset
axins.set_facecolor("#f5f5f5")

# Plot the KML layers in the inset
for layer_name, info in study_layers.items():

    linestyle = ":" if layer_name == "East_River" else "-"

    plot_kml_layer(
        ax=axins,
        gdf=info["gdf"],
        color=info["color"],
        linewidth=2.5,
        linestyle=linestyle,
        zorder=20
    )

# Plot SAIL locations in the inset

# locations_gdf.plot(
#     ax=axins,
#     facecolor="none",
#     color=["black", "magenta"],
#     linewidth=3.50,
#     markersize=65,
#     zorder=25
# )

# Black location as a circle
locations_gdf[locations_gdf["color"] == "black"].plot(
    ax=axins,
    marker="o",
    facecolor="black",
    edgecolor="black",
    linewidth=3.50,
    markersize=85,
    zorder=25
)

# Magenta location as a star
locations_gdf[locations_gdf["color"] == "magenta"].plot(
    ax=axins,
    marker="*",
    facecolor="magenta",
    edgecolor="magenta",
    linewidth=1.0,
    markersize=120,
    zorder=26
)

# Inset extent based on KML and site locations
all_bounds = []

for info in study_layers.values():
    all_bounds.append(info["gdf"].total_bounds)

all_bounds.append(locations_gdf.total_bounds)

ixmin = min(b[0] for b in all_bounds)
iymin = min(b[1] for b in all_bounds)
ixmax = max(b[2] for b in all_bounds)
iymax = max(b[3] for b in all_bounds)

ixpad = (ixmax - ixmin) * 0.12
iypad = (iymax - iymin) * 0.12

axins.set_xlim(ixmin - ixpad, ixmax + ixpad)
axins.set_ylim(iymin - iypad, iymax + iypad)

###################################################
###################################################
# axins.set_title("UCRB and East River Boundaries", fontsize=11)
# axins.tick_params(axis="both", labelsize=8)
# axins.xaxis.set_major_formatter(FuncFormatter(lon_formatter))
# axins.yaxis.set_major_formatter(FuncFormatter(lat_formatter))
# axins.grid(True, alpha=0.3)


axins.set_title("East River Study Area with SAIL Location", fontsize=11, fontweight="bold")
# axins.set_xlabel("Longitude", fontsize=9)
# axins.set_ylabel("Latitude", fontsize=9)
axins.tick_params(axis="both", labelsize=8)
axins.xaxis.set_major_formatter(FuncFormatter(lon_formatter_2dp))
axins.yaxis.set_major_formatter(FuncFormatter(lat_formatter_2dp))

# Make the grid spacing larger
axins.xaxis.set_major_locator(MultipleLocator(0.06))
axins.yaxis.set_major_locator(MultipleLocator(0.06))

# Make grid lines more visible
axins.grid(
    True,
    alpha=0.6,
    linewidth=1.0,
    linestyle="--"
)
# axins.grid(True, alpha=0.3)
###################################################
###################################################

# Draw a rectangle on main map showing the inset area
rect = Rectangle(
    (ixmin - ixpad, iymin - iypad),
    (ixmax - ixmin) + 2 * ixpad,
    (iymax - iymin) + 2 * iypad,
    linewidth=3.0,
    edgecolor="maroon",
    facecolor="none",
    linestyle="-",
    zorder=26
)

ax.add_patch(rect)


# ============================================================
# Add two dashed lines from the star to the inset (zoom area)
# ============================================================

con1 = ConnectionPatch(
    xyA=(sail_center.x, sail_center.y), coordsA=ax.transData,
    xyB=(0.0, 0.0), coordsB=axins.transAxes,
    axesA=ax, axesB=axins,
    color="black",
    linewidth=1.50,
    linestyle="--",
    zorder=27
)

con2 = ConnectionPatch(
    xyA=(sail_center.x, sail_center.y), coordsA=ax.transData,
    xyB=(0.0, 1.0), coordsB=axins.transAxes,
    axesA=ax, axesB=axins,
    color="black",
    linewidth=1.50,
    linestyle="--",
    zorder=27
)

fig.add_artist(con1)
fig.add_artist(con2)


# ============================================================
# Add north arrow to the zoomed inset
# ============================================================

axins.annotate(
    "",
    xy=(0.89, 0.87),       # arrow head
    xytext=(0.89, 0.69),   # arrow tail
    xycoords="axes fraction",
    arrowprops=dict(
        facecolor="black",
        edgecolor="black",
        width=6,
        headwidth=18,
        headlength=35
    ),
    zorder=50
)

axins.text(
    0.89, 0.89,
    "N",
    transform=axins.transAxes,
    ha="center",
    va="bottom",
    fontsize=25,
    fontweight="bold",
    color="black",
    zorder=51,
    bbox=dict(facecolor="white", edgecolor="none", alpha=0.8, pad=1.5)
)

# ============================================================
# Add scale bar to the zoomed inset
# ============================================================

# Choose scale bar length in km
scale_km = 2.0

# Mid-latitude of inset, used to convert longitude degrees to km
mid_lat = 0.5 * ((iymin - iypad) + (iymax + iypad))

# Approximate km per degree longitude at this latitude
km_per_deg_lon = 111.32 * math.cos(math.radians(mid_lat))

# Convert desired scale length from km to degrees longitude
scale_deg = scale_km / km_per_deg_lon

# Position the scale bar near the lower-left of the inset
x0 = (ixmin - ixpad) + 0.08 * ((ixmax + ixpad) - (ixmin - ixpad))
y0 = (iymin - iypad) + 0.08 * ((iymax + iypad) - (iymin - iypad))

# Draw horizontal scale bar
axins.plot(
    [x0, x0 + scale_deg],
    [y0, y0],
    color="black",
    linewidth=3,
    zorder=50
)

# End ticks
tick_h = 0.015 * ((iymax + iypad) - (iymin - iypad))

axins.plot(
    [x0, x0],
    [y0 - tick_h, y0 + tick_h],
    color="black",
    linewidth=3,
    zorder=50
)

axins.plot(
    [x0 + scale_deg, x0 + scale_deg],
    [y0 - tick_h, y0 + tick_h],
    color="black",
    linewidth=3,
    zorder=50
)

# Scale label
axins.text(
    x0 + scale_deg / 2,
    y0 + 1.8 * tick_h,
    f"{scale_km:.0f} km",
    ha="center",
    va="bottom",
    fontsize=11,
    fontweight="bold",
    color="black",
    zorder=51,
    bbox=dict(facecolor="white", edgecolor="none", alpha=0.8, pad=1.2)
)


# ============================================================
# Add north arrow to the main plot
# ============================================================

ax.annotate(
    "",
    xy=(0.92, 0.08),       # arrow head
    xytext=(0.92, 0.01),   # arrow tail
    xycoords="axes fraction",
    arrowprops=dict(
        facecolor="black",
        edgecolor="black",
        width=6,
        headwidth=20,
        headlength=35
    ),
    zorder=50
)

ax.text(
    0.92, 0.08,
    "N",
    transform=ax.transAxes,
    ha="center",
    va="bottom",
    fontsize=30,
    fontweight="bold",
    color="black",
    zorder=51,
    bbox=dict(facecolor="white", edgecolor="none", alpha=0.8, pad=1.5)
)

# ============================================================
# Axes formatting
# ============================================================

ax.set_title(
    "Upper Colorado River Basin (UCRB) within Colorado and Surrounding States",
    fontsize=16, fontweight="bold"
)

ax.set_xlabel("Longitude", fontsize=13)
ax.set_ylabel("Latitude", fontsize=13)

ax.xaxis.set_major_formatter(FuncFormatter(lon_formatter))
ax.yaxis.set_major_formatter(FuncFormatter(lat_formatter))

ax.tick_params(axis="both", labelsize=11)

ax.grid(True, alpha=0.25)

ax.set_aspect("equal")


# ============================================================
# Legend
# ============================================================
legend_handles = [
    Line2D([0], [0], color="black", lw=1.2, linestyle="-.", label="State boundary"),
    Line2D([0], [0], color="purple", lw=2.5, linestyle="--", label="UCRB boundary"),
    Line2D([0], [0], color="red", lw=2.5, label="ERB boundary"),
    Line2D([0], [0], color="blue", lw=2.5, linestyle=":", label="East River Watershed boundary"),
    Line2D(
        [0], [0],
        marker="s",
        color="w",
        markerfacecolor="none",
        markeredgecolor="maroon",
        markeredgewidth=3.30,
        markersize=10,
        label="ERB location"
    ),
    Line2D(
        [0], [0],
        marker="o",
        color="w",
        markerfacecolor="black",
        markeredgecolor="black",
        markeredgewidth=3.30,
        markersize=8,
        label="SAIL site"
    ),
    Line2D(
        [0], [0],
        marker="*",
        color="w",
        markerfacecolor="magenta",
        markeredgecolor="magenta",
        markeredgewidth=1.0,
        markersize=15,
        label="Field pyranometer"
    )
]

ax.legend(
    handles=legend_handles,
    loc="lower left",
    fontsize=10,
    frameon=True,
    facecolor="white",
    framealpha=1.0
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# KML file path and layer names
# ============================================================

kml_path = "/bsuhome/tnde/geoscience/albedo_downscaling/shape_files/East_River.kml"

kml_layers = {
    "East_River.kml": "red",   # larger boundary
    "East_River": "blue"       # smaller boundary
}

# ============================================================
# Read both KML layers in lon/lat
# ============================================================
study_layers = {}

for layer_name, color in kml_layers.items():

    gdf = gpd.read_file(kml_path, layer=layer_name)

    gdf = gdf[
        gdf.geometry.notnull() & ~gdf.geometry.is_empty
    ].copy()

    # KML is already lon/lat.
    # Only assign CRS if GeoPandas did not detect it.
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326")

    # Keep everything in lon/lat for the regional map
    gdf = gdf.to_crs("EPSG:4326")

    study_layers[layer_name] = {
        "gdf": gdf,
        "color": color
    }

    print(f"\nLayer: {layer_name}")
    print("CRS:", gdf.crs)
    print("Geometry types:", gdf.geometry.geom_type.unique())
    print("Bounds:", gdf.total_bounds)

# ============================================================
# Read UCRB polygon from separate KML file
# ============================================================
ucrb_kml_path = "/bsuhome/tnde/geoscience/albedo_downscaling/shape_files/Colorado_River_Basin_Hydrological_Boundaries.kml"

ucrb_gdf = gpd.read_file(ucrb_kml_path)

ucrb_gdf = ucrb_gdf[
    ucrb_gdf.geometry.notnull() & ~ucrb_gdf.geometry.is_empty
].copy()

if ucrb_gdf.crs is None:
    ucrb_gdf = ucrb_gdf.set_crs("EPSG:4326")

ucrb_gdf = ucrb_gdf.to_crs("EPSG:4326")

# Select polygon 8.
# Python indexing starts from 0, so polygon 8 is index 7.
ucrb_polygon = ucrb_gdf.iloc[[7]].copy()

print("\nSelected UCRB polygon:")
print("CRS:", ucrb_polygon.crs)
print("Geometry types:", ucrb_polygon.geometry.geom_type.unique())
print("Bounds:", ucrb_polygon.total_bounds)
    
# ============================================================
# SAIL locations in lon/lat
# ============================================================

# 38°57'22.35"N, 106°59'16.66"W
sail_road_lon = -106.987961
sail_road_lat = 38.956208

# 38°57'22.99"N, 106°59'8.79"W
sail_hill_lon = -106.985775
sail_hill_lat = 38.956386

locations_gdf = gpd.GeoDataFrame(
    {
        "name": [
            "SAIL site near County Road 317",
            "Field pyranometer on adjacent hill"
        ],
        "color": [
            "black",
            "magenta"
        ]
    },
    geometry=[
        Point(sail_road_lon, sail_road_lat),
        Point(sail_hill_lon, sail_hill_lat)
    ],
    crs="EPSG:4326"
)

# ============================================================
# Reproject KML layers and locations to Web Mercator for inset basemap
# ============================================================

inset_crs = "EPSG:3857"

study_layers_3857 = {}

for layer_name, info in study_layers.items():
    study_layers_3857[layer_name] = {
        "gdf": info["gdf"].to_crs(inset_crs),
        "color": info["color"]
    }

locations_3857 = locations_gdf.to_crs(inset_crs)

# ============================================================
# Read Colorado and surrounding state boundaries
# ============================================================

states_url = "https://www2.census.gov/geo/tiger/GENZ2023/shp/cb_2023_us_state_500k.zip"

states = gpd.read_file(states_url).to_crs("EPSG:4326")

state_names = [
    "Colorado",
    "Utah",
    "Wyoming",
    "Nebraska",
    "Kansas",
    "Oklahoma",
    "New Mexico",
    "Arizona",
    "Texas"
]

states_region = states[states["NAME"].isin(state_names)].copy()
colorado = states_region[states_region["NAME"] == "Colorado"].copy()


# ============================================================
# Helper functions
# ============================================================

def plot_kml_layer(ax, gdf, color, linewidth=2.5, linestyle="-", zorder=10):
    """
    Plot one KML layer.
    Polygons are plotted as boundaries.
    Lines are plotted as lines.
    Points are plotted as points.
    """

    polygon_types = ["Polygon", "MultiPolygon"]
    line_types = ["LineString", "MultiLineString", "LinearRing"]
    point_types = ["Point", "MultiPoint"]

    poly_gdf = gdf[gdf.geometry.geom_type.isin(polygon_types)]
    if not poly_gdf.empty:
        poly_gdf.boundary.plot(
            ax=ax,
            color=color,
            linewidth=linewidth,
            linestyle=linestyle,
            zorder=zorder
        )

    line_gdf = gdf[gdf.geometry.geom_type.isin(line_types)]
    if not line_gdf.empty:
        line_gdf.plot(
            ax=ax,
            color=color,
            linewidth=linewidth,
            linestyle=linestyle,
            zorder=zorder
        )

    point_gdf = gdf[gdf.geometry.geom_type.isin(point_types)]
    if not point_gdf.empty:
        point_gdf.plot(
            ax=ax,
            color=color,
            markersize=35,
            zorder=zorder
        )

    other_gdf = gdf[
        ~gdf.geometry.geom_type.isin(polygon_types + line_types + point_types)
    ]
    if not other_gdf.empty:
        other_gdf.plot(
            ax=ax,
            facecolor="none",
            edgecolor=color,
            color=color,
            linewidth=linewidth,
            linestyle=linestyle,
            zorder=zorder
        )


def lon_formatter(x, pos):
    return f"{abs(x):.0f}$^\circ$W" if x < 0 else f"{x:.0f}$^\circ$E"
def lat_formatter(y, pos):
    return f"{abs(y):.0f}$^\circ$S" if y < 0 else f"{y:.0f}$^\circ$N"

def lon_formatter_2dp(x, pos):
    return f"{abs(x):.2f}$^\circ$W" if x < 0 else f"{x:.2f}$^\circ$E"
def lat_formatter_2dp(y, pos):
    return f"{abs(y):.2f}$^\circ$S" if y < 0 else f"{y:.2f}$^\circ$N"

# Add 2-decimal tick formatters for the inset in Web Mercator
merc_to_geo = Transformer.from_crs("EPSG:3857", "EPSG:4326", always_xy=True)
def lon_formatter_2dp_3857(x, pos):
    lon, _ = merc_to_geo.transform(x, 0)
    return f"{abs(lon):.2f}$^\circ$W" if lon < 0 else f"{lon:.2f}$^\circ$E"
def lat_formatter_2dp_3857(y, pos):
    _, lat = merc_to_geo.transform(0, y)
    return f"{abs(lat):.2f}$^\circ$S" if lat < 0 else f"{lat:.2f}$^\circ$N"

# ============================================================
# Create main figure
# ============================================================

fig, ax = plt.subplots(figsize=(11, 10))


# ============================================================
# Plot surrounding states
# ============================================================

states_region.plot(
    ax=ax,
    facecolor="#55a87f",
    edgecolor="black",
    linestyle="-.",
    linewidth=0.8,
    zorder=1
)

# Highlight Colorado slightly
colorado.plot(
    ax=ax,
    facecolor="#4b9c78",
    edgecolor="black",
    linestyle="-.",
    linewidth=1.0,
    zorder=2
)

# ============================================================
# Add state labels
# ============================================================

for _, row in states_region.iterrows():
    point = row.geometry.representative_point()

    text = ax.text(
        point.x,
        point.y,
        row["NAME"],
        fontsize=13,
        color="white",
        ha="center",
        va="center",
        zorder=5
    )

    text.set_path_effects([
        pe.withStroke(linewidth=2.5, foreground="black")
    ])


# # ============================================================
# # Plot KML layers on main map
# # ============================================================
# for layer_name, info in study_layers.items():

#     linestyle = "--" if layer_name == "East_River" else "-"

#     plot_kml_layer(
#         ax=ax,
#         gdf=info["gdf"],
#         color=info["color"],
#         linewidth=2.2,
#         linestyle=linestyle,
#         zorder=20
#     )

# ============================================================
# Plot UCRB polygon on main map only
# ============================================================
plot_kml_layer(
    ax=ax,
    gdf=ucrb_polygon,
    color="purple",
    linewidth=2.5,
    linestyle="--",
    zorder=20
)

# # ============================================================
# # Plot East River boundary on main map
# # ============================================================
# if "East_River" in study_layers:

#     plot_kml_layer(
#         ax=ax,
#         gdf=study_layers["East_River"]["gdf"],
#         color="blue",
#         linewidth=2.2,
#         linestyle="--",
#         zorder=21
#     )

# # ============================================================
# # Plot SAIL locations on main map
# # ============================================================
# locations_gdf.plot(
#     ax=ax,
#     color=["blue", "magenta"],
#     markersize=55,
#     zorder=25
# )

# Add a larger star so the area is visible at state scale
sail_center = locations_gdf.geometry.unary_union.centroid

# ax.scatter(
#     sail_center.x,
#     sail_center.y,
#     s=650,
#     marker="*",
#     color="yellow",
#     edgecolor="black",
#     linewidth=1.0,
#     zorder=30
# )

text = ax.annotate(
    "ERB",
    xy=(sail_center.x, sail_center.y),
    xytext=(-13, 8),
    textcoords="offset points",
    fontsize=12,
    fontweight="bold",
    color="white",
    zorder=31
)

text.set_path_effects([
    pe.withStroke(linewidth=2.5, foreground="black")
])


# ============================================================
# Set main map extent
# ============================================================

xmin, ymin, xmax, ymax = states_region.total_bounds

xpad = (xmax - xmin) * 0.05
ypad = (ymax - ymin) * 0.05

ax.set_xlim(xmin - xpad, xmax + xpad)
ax.set_ylim(ymin - ypad, ymax + ypad)

# ============================================================
# Add zoomed inset for the KML boundaries with Esri World Imagery
# ============================================================

# axins = inset_axes(
#     ax,
#     width="38%",
#     height="38%",
#     loc="upper right",
#     borderpad=2.50
# )

axins = inset_axes(
    ax,
    width="47%",
    height="43%",
    loc="upper right",
    bbox_to_anchor=(0.068, 0.0, 1, 1),  # shift inset slightly to the right
    bbox_transform=ax.transAxes,
    borderpad=2.0
)

# Light background for inset
axins.set_facecolor("#f5f5f5")

# Inset extent based on KML and site locations
all_bounds = []

for info in study_layers.values():
    all_bounds.append(info["gdf"].total_bounds)

all_bounds.append(locations_gdf.total_bounds)

ixmin = min(b[0] for b in all_bounds)
iymin = min(b[1] for b in all_bounds)
ixmax = max(b[2] for b in all_bounds)
iymax = max(b[3] for b in all_bounds)

ixpad = (ixmax - ixmin) * 0.12
iypad = (iymax - iymin) * 0.12

axins.set_xlim(ixmin - ixpad, ixmax + ixpad)
axins.set_ylim(iymin - iypad, iymax + iypad)

# ============================================================
# Add Esri World Imagery to the zoomed inset only
# ============================================================

ctx.add_basemap(
    axins,
    source=ctx.providers.Esri.WorldImagery,
    crs="EPSG:4326",
    zoom=13,
    attribution_size=5
)

# Keep inset limits unchanged after adding basemap
axins.set_xlim(ixmin - ixpad, ixmax + ixpad)
axins.set_ylim(iymin - iypad, iymax + iypad)

# Plot the KML layers in the inset
for layer_name, info in study_layers.items():

    linestyle = ":" if layer_name == "East_River" else "-"

    plot_kml_layer(
        ax=axins,
        gdf=info["gdf"],
        color=info["color"],
        linewidth=2.5,
        linestyle=linestyle,
        zorder=20
    )

# # Plot SAIL locations in the inset

# locations_gdf.plot(
#     ax=axins,
#     facecolor="none",
#     color=["blue", "magenta"],
#     linewidth=3.50,
#     markersize=65,
#     zorder=25
# )

# Black location as a circle
locations_gdf[locations_gdf["color"] == "black"].plot(
    ax=axins,
    marker="o",
    facecolor="blue",
    edgecolor="blue",
    linewidth=3.50,
    markersize=85,
    zorder=25
)

# Magenta location as a star
locations_gdf[locations_gdf["color"] == "magenta"].plot(
    ax=axins,
    marker="*",
    facecolor="magenta",
    edgecolor="magenta",
    linewidth=1.0,
    markersize=120,
    zorder=26
)

###################################################
###################################################
# axins.set_title("UCRB and East River Boundaries", fontsize=11)
# axins.tick_params(axis="both", labelsize=8)
# axins.xaxis.set_major_formatter(FuncFormatter(lon_formatter))
# axins.yaxis.set_major_formatter(FuncFormatter(lat_formatter))
# axins.grid(True, alpha=0.3)


axins.set_title("East River Study Area with SAIL Location", fontsize=11, fontweight="bold")
# axins.set_xlabel("Longitude", fontsize=9)
# axins.set_ylabel("Latitude", fontsize=9)
axins.tick_params(axis="both", labelsize=8)
axins.xaxis.set_major_formatter(FuncFormatter(lon_formatter_2dp))
axins.yaxis.set_major_formatter(FuncFormatter(lat_formatter_2dp))

# Make the grid spacing larger
axins.xaxis.set_major_locator(MultipleLocator(0.06))
axins.yaxis.set_major_locator(MultipleLocator(0.06))

# Make grid lines more visible
axins.grid(
    True,
    alpha=0.6,
    linewidth=1.0,
    linestyle="--"
)
# axins.grid(True, alpha=0.3)
###################################################
###################################################

# Draw a rectangle on main map showing the inset area
rect = Rectangle(
    (ixmin - ixpad, iymin - iypad),
    (ixmax - ixmin) + 2 * ixpad,
    (iymax - iymin) + 2 * iypad,
    linewidth=3.0,
    edgecolor="maroon",
    facecolor="none",
    linestyle="-",
    zorder=26
)

ax.add_patch(rect)


# ============================================================
# Add two dashed lines from the star to the inset (zoom area)
# ============================================================

con1 = ConnectionPatch(
    xyA=(sail_center.x, sail_center.y), coordsA=ax.transData,
    xyB=(0.0, 0.0), coordsB=axins.transAxes,
    axesA=ax, axesB=axins,
    color="black",
    linewidth=1.50,
    linestyle="--",
    zorder=27
)

con2 = ConnectionPatch(
    xyA=(sail_center.x, sail_center.y), coordsA=ax.transData,
    xyB=(0.0, 1.0), coordsB=axins.transAxes,
    axesA=ax, axesB=axins,
    color="black",
    linewidth=1.50,
    linestyle="--",
    zorder=27
)

fig.add_artist(con1)
fig.add_artist(con2)


# ============================================================
# Add north arrow to the zoomed inset
# ============================================================

axins.annotate(
    "",
    xy=(0.89, 0.87),       # arrow head
    xytext=(0.89, 0.69),   # arrow tail
    xycoords="axes fraction",
    arrowprops=dict(
        facecolor="black",
        edgecolor="black",
        width=6,
        headwidth=18,
        headlength=35
    ),
    zorder=50
)

axins.text(
    0.89, 0.89,
    "N",
    transform=axins.transAxes,
    ha="center",
    va="bottom",
    fontsize=25,
    fontweight="bold",
    color="black",
    zorder=51,
    bbox=dict(facecolor="white", edgecolor="none", alpha=0.8, pad=1.5)
)

# ============================================================
# Add scale bar to the zoomed inset
# ============================================================

# Choose scale bar length in km
scale_km = 2.0

# Mid-latitude of inset, used to convert longitude degrees to km
mid_lat = 0.5 * ((iymin - iypad) + (iymax + iypad))

# Approximate km per degree longitude at this latitude
km_per_deg_lon = 111.32 * math.cos(math.radians(mid_lat))

# Convert desired scale length from km to degrees longitude
scale_deg = scale_km / km_per_deg_lon

# Position the scale bar near the lower-left of the inset
x0 = (ixmin - ixpad) + 0.08 * ((ixmax + ixpad) - (ixmin - ixpad))
y0 = (iymin - iypad) + 0.08 * ((iymax + iypad) - (iymin - iypad))

# Draw horizontal scale bar
axins.plot(
    [x0, x0 + scale_deg],
    [y0, y0],
    color="black",
    linewidth=3,
    zorder=50
)

# End ticks
tick_h = 0.015 * ((iymax + iypad) - (iymin - iypad))

axins.plot(
    [x0, x0],
    [y0 - tick_h, y0 + tick_h],
    color="black",
    linewidth=3,
    zorder=50
)

axins.plot(
    [x0 + scale_deg, x0 + scale_deg],
    [y0 - tick_h, y0 + tick_h],
    color="black",
    linewidth=3,
    zorder=50
)

# Scale label
axins.text(
    x0 + scale_deg / 2,
    y0 + 1.8 * tick_h,
    f"{scale_km:.0f} km",
    ha="center",
    va="bottom",
    fontsize=11,
    fontweight="bold",
    color="black",
    zorder=51,
    bbox=dict(facecolor="white", edgecolor="none", alpha=0.8, pad=1.2)
)


# ============================================================
# Add north arrow to the main plot
# ============================================================

ax.annotate(
    "",
    xy=(0.92, 0.08),       # arrow head
    xytext=(0.92, 0.01),   # arrow tail
    xycoords="axes fraction",
    arrowprops=dict(
        facecolor="black",
        edgecolor="black",
        width=6,
        headwidth=20,
        headlength=35
    ),
    zorder=50
)

ax.text(
    0.92, 0.08,
    "N",
    transform=ax.transAxes,
    ha="center",
    va="bottom",
    fontsize=30,
    fontweight="bold",
    color="black",
    zorder=51,
    bbox=dict(facecolor="white", edgecolor="none", alpha=0.8, pad=1.5)
)

# ============================================================
# Axes formatting
# ============================================================

ax.set_title(
    "Upper Colorado River Basin (UCRB) within Colorado and Surrounding States",
    fontsize=16, fontweight="bold"
)

ax.set_xlabel("Longitude", fontsize=13)
ax.set_ylabel("Latitude", fontsize=13)

ax.xaxis.set_major_formatter(FuncFormatter(lon_formatter))
ax.yaxis.set_major_formatter(FuncFormatter(lat_formatter))

ax.tick_params(axis="both", labelsize=11)

ax.grid(True, alpha=0.25)

ax.set_aspect("equal")

# ============================================================
# Legend
# ============================================================
legend_handles = [
    Line2D([0], [0], color="black", lw=1.2, linestyle="-.", label="State boundary"),
    Line2D([0], [0], color="purple", lw=2.5, linestyle="--", label="UCRB boundary"),
    Line2D([0], [0], color="red", lw=2.5, label="ERB boundary"),
    Line2D([0], [0], color="blue", lw=2.5, linestyle=":", label="East River Watershed boundary"),
    Line2D(
        [0], [0],
        marker="s",
        color="w",
        markerfacecolor="none",
        markeredgecolor="maroon",
        markeredgewidth=3.30,
        markersize=10,
        label="ERB location"
    ),
    Line2D(
        [0], [0],
        marker="o",
        color="w",
        markerfacecolor="blue",
        markeredgecolor="blue",
        markeredgewidth=3.30,
        markersize=8,
        label="SAIL site"
    ),
    Line2D(
        [0], [0],
        marker="*",
        color="w",
        markerfacecolor="magenta",
        markeredgecolor="magenta",
        markeredgewidth=1.0,
        markersize=15,
        label="Field pyranometer"
    )
]

ax.legend(
    handles=legend_handles,
    loc="lower left",
    fontsize=10,
    frameon=True,
    facecolor="white",
    framealpha=1.0
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# KML file path and layer names
# ============================================================

kml_path = "/bsuhome/tnde/geoscience/albedo_downscaling/shape_files/East_River.kml"

# New KML file with all Colorado River Basin polygons
ucrb_kml_path = "/bsuhome/tnde/geoscience/albedo_downscaling/shape_files/Colorado_River_Basin_Hydrological_Boundaries.kml"

kml_layers = {
    "East_River.kml": "red",
    "East_River": "blue"
}

# ============================================================
# Read both East River KML layers
# ============================================================
study_layers = {}

for layer_name, color in kml_layers.items():

    gdf = gpd.read_file(kml_path, layer=layer_name)

    gdf = gdf[
        gdf.geometry.notnull() & ~gdf.geometry.is_empty
    ].copy()

    # KML is already lon/lat.
    # Only assign CRS if GeoPandas did not detect it.
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326")

    # Folium needs lon/lat, EPSG:4326
    gdf = gdf.to_crs("EPSG:4326")

    study_layers[layer_name] = {
        "gdf": gdf,
        "color": color
    }

    print(f"\nLayer: {layer_name}")
    print("CRS:", gdf.crs)
    print("Geometry types:", gdf.geometry.geom_type.unique())
    print("Bounds:", gdf.total_bounds)


# ============================================================
# Read new KML file with all polygons
# ============================================================

ucrb_gdf = gpd.read_file(ucrb_kml_path)

ucrb_gdf = ucrb_gdf[
    ucrb_gdf.geometry.notnull() & ~ucrb_gdf.geometry.is_empty
].copy()

# KML is usually lon/lat.
# Only assign CRS if GeoPandas did not detect it.
if ucrb_gdf.crs is None:
    ucrb_gdf = ucrb_gdf.set_crs("EPSG:4326")

# Folium needs lon/lat, EPSG:4326
ucrb_gdf = ucrb_gdf.to_crs("EPSG:4326")

# Keep all polygons. If some are stored as MultiPolygon, this separates them.
ucrb_gdf = ucrb_gdf.explode(index_parts=False).reset_index(drop=True)

print("\nNew KML file: Colorado River Basin Hydrological Boundaries")
print("CRS:", ucrb_gdf.crs)
print("Geometry types:", ucrb_gdf.geometry.geom_type.unique())
print("Number of polygons/features:", len(ucrb_gdf))
print("Bounds:", ucrb_gdf.total_bounds)

    
# ============================================================
# Read state boundaries
# ============================================================

states_url = "https://www2.census.gov/geo/tiger/GENZ2023/shp/cb_2023_us_state_500k.zip"

states = gpd.read_file(states_url).to_crs("EPSG:4326")

# Include all 50 U.S. states
us_state_abbrevs = [
    "AL", "AK", "AZ", "AR", "CA", "CO", "CT", "DE", "FL", "GA",
    "HI", "ID", "IL", "IN", "IA", "KS", "KY", "LA", "ME", "MD",
    "MA", "MI", "MN", "MS", "MO", "MT", "NE", "NV", "NH", "NJ",
    "NM", "NY", "NC", "ND", "OH", "OK", "OR", "PA", "RI", "SC",
    "SD", "TN", "TX", "UT", "VT", "VA", "WA", "WV", "WI", "WY"
]

states_region = states[states["STUSPS"].isin(us_state_abbrevs)].copy()


# ============================================================
# SAIL locations in lon/lat
# ============================================================

# 38°57'22.35"N, 106°59'16.66"W
sail_road_lon = -106.987961
sail_road_lat = 38.956208

# 38°57'22.99"N, 106°59'8.79"W
sail_hill_lon = -106.985775
sail_hill_lat = 38.956386

locations_gdf = gpd.GeoDataFrame(
    {
        "name": [
            "SAIL site near County Road 317",
            "Field instruments on adjacent hill"
        ],
        "color": [
            "black",
            "magenta"
        ]
    },
    geometry=[
        Point(sail_road_lon, sail_road_lat),
        Point(sail_hill_lon, sail_hill_lat)
    ],
    crs="EPSG:4326"
)


# ============================================================
# Compute map center and bounds
# ============================================================

all_bounds = []

for info in study_layers.values():
    all_bounds.append(info["gdf"].total_bounds)

# Include the new KML file bounds
all_bounds.append(ucrb_gdf.total_bounds)

all_bounds.append(locations_gdf.total_bounds)

xmin = min(b[0] for b in all_bounds)
ymin = min(b[1] for b in all_bounds)
xmax = max(b[2] for b in all_bounds)
ymax = max(b[3] for b in all_bounds)

center_lat = (ymin + ymax) / 2
center_lon = (xmin + xmax) / 2


# ============================================================
# Create interactive map
# ============================================================

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=13,
    tiles=None
)


# ============================================================
# Add Esri World Imagery basemap
# ============================================================

esri_world_imagery = ctx.providers.Esri.WorldImagery

folium.TileLayer(
    tiles=esri_world_imagery.build_url(),
    attr=esri_world_imagery.attribution,
    name="Esri World Imagery",
    overlay=False,
    control=True
).add_to(m)


# Optional: add OpenStreetMap as another selectable basemap
folium.TileLayer(
    tiles="OpenStreetMap",
    name="OpenStreetMap",
    overlay=False,
    control=True
).add_to(m)


# ============================================================
# Add state boundaries on top of Esri World Imagery
# ============================================================

folium.GeoJson(
    states_region,
    name="State boundaries",
    style_function=lambda feature: {
        "color": "white",
        "weight": 4,
        "fillOpacity": 0.0,
        "opacity": 0.9
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["NAME"],
        aliases=["State:"],
        sticky=True
    )
).add_to(m)

folium.GeoJson(
    states_region,
    name="State boundaries - black outline",
    style_function=lambda feature: {
        "color": "black",
        "weight": 1.5,
        "fillOpacity": 0.0,
        "opacity": 1.0
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["NAME"],
        aliases=["State:"],
        sticky=True
    )
).add_to(m)


# ============================================================
# Add new KML file as interactive GeoJSON overlay
# ============================================================

folium.GeoJson(
    ucrb_gdf,
    name="Colorado River Basin Hydrological Boundaries",
    style_function=lambda feature: {
        "color": "maroon",
        "weight": 3,
        "fillColor": "maroon",
        "fillOpacity": 0.08,
        "opacity": 1.0
    },
    highlight_function=lambda feature: {
        "weight": 5,
        "opacity": 1.0,
        "fillOpacity": 0.15
    },
    tooltip=folium.GeoJsonTooltip(
        fields=[col for col in ucrb_gdf.columns if col != "geometry"][:3],
        aliases=[col for col in ucrb_gdf.columns if col != "geometry"][:3],
        sticky=True
    ) if len([col for col in ucrb_gdf.columns if col != "geometry"]) > 0 else None
).add_to(m)


# ============================================================
# Add East River KML layers as interactive GeoJSON overlays
# ============================================================

for layer_name, info in study_layers.items():

    gdf = info["gdf"]
    color = info["color"]

    linestyle = "5, 5" if layer_name == "East_River" else None

    folium.GeoJson(
        gdf,
        name=layer_name,
        style_function=lambda feature, color=color, linestyle=linestyle: {
            "color": color,
            "weight": 3,
            "fillColor": color,
            "fillOpacity": 0.05,
            "opacity": 1.0,
            "dashArray": linestyle
        },
        highlight_function=lambda feature: {
            "weight": 5,
            "opacity": 1.0
        },
        tooltip=folium.GeoJsonTooltip(
            fields=[col for col in gdf.columns if col != "geometry"][:3],
            aliases=[col for col in gdf.columns if col != "geometry"][:3],
            sticky=True
        ) if len([col for col in gdf.columns if col != "geometry"]) > 0 else None
    ).add_to(m)


# ============================================================
# Add SAIL location markers
# ============================================================

for _, row in locations_gdf.iterrows():

    lon = row.geometry.x
    lat = row.geometry.y
    name = row["name"]
    color = row["color"]

    folium.CircleMarker(
        location=[lat, lon],
        radius=7,
        color="white",
        weight=2,
        fill=True,
        fill_color=color,
        fill_opacity=1.0,
        popup=folium.Popup(name, max_width=300),
        tooltip=name
    ).add_to(m)


# ============================================================
# Fit map to KML/site layers
# ============================================================

# m.fit_bounds([
#     [ymin, xmin],
#     [ymax, xmax]
# ])

sxmin, symin, sxmax, symax = states_region.total_bounds

m.fit_bounds([
    [symin, sxmin],
    [symax, sxmax]
])


# ============================================================
# Add layer control and save map
# ============================================================

folium.LayerControl(collapsed=False).add_to(m)

output_html = "east_river_interactive_map.html"
m.save(output_html)

print(f"Interactive map saved as: {output_html}")

# Display in Jupyter Notebook
m